# 🌐 Unified Knowledge Base Builder
Takes a list of HTML websites, automatically crawls them to discover all hosted PDFs, downloads them, extracts the text, and converts them into the FAISS and BM25 Vector Databases in one single step.

**No hardcoded PDFs.** The system naturally discovers them from the HTML domains.

In [ ]:
%%writefile /content/pdf_web_crawler.py
#!/usr/bin/env python3
"""
pdf_web_crawler.py
==================
Crawls seed websites → discovers ALL PDFs → downloads them →
rebuilds FAISS + BM25 indexes.

Drive folders:
  FranchiseOps → MyDrive/FranchiseOps_AI/rag_pdfs/
  FreightQuote → MyDrive/FreightQuote_AI/rag_pdfs/
"""

import os
import re
import json
import time
import pickle
import hashlib
import requests
import urllib.parse
import subprocess
import sys

from pathlib import Path
from collections import deque
from datetime import datetime

# Install deps silently
for pkg in ["requests", "beautifulsoup4", "lxml", "tqdm",
            "PyMuPDF", "sentence-transformers", "faiss-cpu", "rank-bm25"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                   capture_output=True)

from bs4 import BeautifulSoup
from tqdm.auto import tqdm

# ─── Drive folder names ───────────────────────────────────────────────────────

FQ_DRIVE_FOLDER = "FreightQuote_AI"

# ─── Crawler settings ─────────────────────────────────────────────────────────
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                   "Chrome/120.0.0.0 Safari/537.36"),
    "Accept": "text/html,application/xhtml+xml,*/*;q=0.8",
}
PDF_HEADERS = {
    "User-Agent": HEADERS["User-Agent"],
    "Accept": "application/pdf,*/*",
}

MAX_CRAWL_DEPTH   = 2
MAX_PDFS_PER_SEED = 50
DOWNLOAD_TIMEOUT  = 25
REQUEST_TIMEOUT   = 15
DELAY_BETWEEN     = 0.4
CHUNK_WORDS       = 400
CHUNK_OVERLAP     = 50

# ─────────────────────────────────────────────────────────────────────────────
# SEED URLS
# ─────────────────────────────────────────────────────────────────────────────



FREIGHT_SEEDS = [
    ("Web Source", "https://www.iata.org/en/programs/cargo/security"),
    ("Web Source", "https://www.iata.org/en/programs/cargo/live-animals"),
    ("Web Source", "https://www.classnk.or.jp/hp/en/index.html"),
    ("Web Source", "https://www.trade.gov/trade-data-and-analysis"),
    ("Web Source", "http://www.wcoomd.org/en/topics/nomenclature/overview/what-is-the-harmonized-system.aspx"),
    ("Web Source", "https://www.fmc.gov/shipping-guidance"),
    ("Web Source", "https://www.cbp.gov/trade/basic-import-export"),
    ("Web Source", "https://www.worldbank.org/en/topic/trade"),
    ("Web Source", "https://marad.dot.gov"),
    ("Web Source", "https://www.gard.no/web/updates"),
    ("Web Source", "https://www.investopedia.com/terms/f/freight.asp"),
    ("Web Source", "https://www.cbp.gov/trade/automated"),
    ("Web Source", "https://www.trade.gov/customs-broker"),
    ("Web Source", "https://www.marad.dot.gov"),
    ("Web Source", "https://www.customs.gov.sg/businesses/importing-goods/overview"),
    ("Web Source", "https://www.gov.uk/guidance/customs-declaration-service"),
    ("Web Source", "https://www.joc.com"),
    ("Web Source", "https://www.gov.uk/import-goods-into-uk"),
    ("Web Source", "https://www.investopedia.com/terms/p/profit-margin.asp"),
    ("Web Source", "https://taxation-customs.ec.europa.eu/customs-4/union-customs-code_en"),
    ("Web Source", "https://www.bis.doc.gov/index.php/regulations/export-administration-regulations-ear"),
    ("Web Source", "https://unctad.org/topic/trade-analysis"),
    ("Web Source", "https://www.gov.uk/guidance/intrastat"),
    ("Web Source", "https://www.gov.uk/government/collections/uk-trade-tariff"),
    ("Web Source", "https://www.iata.org/en/programs/cargo/dgr"),
    ("Web Source", "https://www.customs.gov.sg/businesses/exporting-goods/overview"),
    ("Web Source", "https://www.cbp.gov/trade/priority-issues"),
    ("Web Source", "https://www.gov.uk/guidance/pay-less-import-duty-and-vat-when-re-importing-goods-to-the-uk-and-eu"),
    ("Web Source", "http://www.wcoomd.org/en/topics/facilitation/overview/overview-of-trade-facilitation.aspx"),
    ("Web Source", "https://www.abf.gov.au/importing-exporting-and-manufacturing/importing"),
    ("Web Source", "https://taxation-customs.ec.europa.eu/customs-4/customs-procedures-import_en"),
    ("Web Source", "https://www.wto.org/english/tratop_e/tradfa_e/tradfa_e.htm"),
    ("Web Source", "https://unctad.org/topic/trade-logistics"),
    ("Web Source", "https://www.cbp.gov/trade/trade-community"),
    ("Web Source", "https://www.intercargo.org"),
    ("Web Source", "https://taxation-customs.ec.europa.eu/customs-4/prohibitions-and-restrictions_en"),
    ("Web Source", "https://www.fmc.gov/regulations"),
    ("Web Source", "https://taxation-customs.ec.europa.eu/customs-4/customs-procedures-export_en"),
    ("Web Source", "http://www.wcoomd.org/en/topics/enforcement-and-compliance.aspx"),
    ("Web Source", "https://www.abf.gov.au/importing-exporting-and-manufacturing/exporting"),
    ("Web Source", "https://www.gov.uk/export-goods"),
    ("Web Source", "https://www.lr.org/en/maritime"),
    ("Web Source", "https://www.worldbank.org/en/topic/transport/overview"),
    ("Web Source", "https://www.customs.gov.sg/businesses/customs-schemes-licences-framework/overview"),
    ("Web Source", "https://iccwbo.org/business-solutions/incoterms-rules/incoterms-2020"),
    ("Web Source", "https://www.icegate.gov.in"),
    ("Web Source", "https://www.wto.org/english/tratop_e/tariffs_e/tariffs_e.htm"),
    ("Web Source", "https://www.abf.gov.au/importing-exporting-and-manufacturing/tariff-classification"),
    ("Web Source", "https://www.logisticsmgmt.com"),
    ("Web Source", "https://www.freightos.com/freight-resources/freight-rate"),
    ("Web Source", "https://www.wto.org/english/thewto_e/whatis_e/tif_e/agrm8_e.htm"),
    ("Web Source", "https://www.trade.gov/export-solutions"),
    ("Web Source", "https://www.imo.org/en/OurWork/Safety/Pages/default.aspx"),
    ("Web Source", "https://www.fmc.gov/about-the-fmc"),
    ("Web Source", "http://english.customs.gov.cn"),
    ("Web Source", "https://www.supplychaindigital.com"),
    ("Web Source", "https://www.bimco.org/contracts-and-clauses/contracts"),
    ("Web Source", "https://www.bis.doc.gov/index.php/licensing"),
    ("Maritime Law",      "https://www.imo.org/en/KnowledgeCentre/IndexofIMOResolutions/"),
    ("Maritime Law",      "https://www.imo.org/en/OurWork/Safety/Pages/Default.aspx"),
    ("Shipping Reports",  "https://unctad.org/topic/transport-and-trade-logistics/review-of-maritime-transport"),
    ("Shipping Reports",  "https://unctad.org/publications?field_topics_target_id=78"),
    ("Customs",           "https://www.wcoomd.org/en/topics/nomenclature/resources.aspx"),
    ("Customs",           "https://www.wcoomd.org/en/topics/valuation/resources.aspx"),
    ("INCOTERMS",         "https://iccwbo.org/resources-for-business/incoterms-rules/"),
    ("US Customs",        "https://www.cbp.gov/trade/publications"),
    ("India Trade",       "https://www.dgft.gov.in/CP/"),
    ("India Trade",       "https://www.cbic.gov.in/resources//htdocs-cbec/customs/"),
    ("India Trade",       "https://shipmin.gov.in/en/publications"),
    ("India Trade",       "https://www.jnport.gov.in/page/publications"),
    ("Port Operations",   "https://unctad.org/topic/transport-and-trade-logistics/port-management"),
    ("Port Operations",   "https://www.mpa.gov.sg/publications/statistics"),
    ("Freight Pricing",   "https://unctad.org/topic/transport-and-trade-logistics/liner-shipping"),
    ("Freight Pricing",   "https://lpi.worldbank.org/"),
    ("Marine Insurance",  "https://www.ukpandi.com/knowledge-publications/"),
    ("Marine Insurance",  "https://www.gard.no/web/publications"),
    ("Marine Insurance",  "https://www.iumi.com/publications/"),
    ("Weather Risk",      "https://library.wmo.int/records?ln=en&p=marine&action_search=Search&c="),
    ("Digital Logistics", "https://unctad.org/topic/transport-and-trade-logistics/port-technology"),
    ("Digital Logistics", "https://dcsa.org/resources/"),
    ("Environmental",     "https://www.imo.org/en/OurWork/Environment/Pages/Default.aspx"),
    ("Environmental",     "https://www.dnv.com/maritime/publications/index.html"),
    ("Trade Finance",     "https://iccwbo.org/resources-for-business/trade-finance/"),
    ("Trade Finance",     "https://www.adb.org/publications?subject=trade-finance"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=maritime+freight+pricing+machine+learning"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=port+congestion+prediction"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=customs+risk+classification"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=vessel+route+optimization"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=supply+chain+disruption+prediction"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=freight+rate+forecasting"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=RAG+large+language+model+logistics"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=digital+twin+shipping"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=container+shipping+deep+learning"),
    ("Shipping Bodies",   "https://www.ics-shipping.org/resources/"),
    ("Shipping Bodies",   "https://www.bimco.org/education-and-training/publications"),
    ("India Finance",     "https://rbidocs.rbi.org.in/rdocs/Publications/"),
    ("EU Customs",        "https://taxation-customs.ec.europa.eu/customs-4_en"),
    ("WTO",               "https://www.wto.org/english/res_e/publications_e/"),
    ("ICS",               "https://www.ics-shipping.org/resources/"),
    ("FONASBA",           "https://www.fonasba.com/publications"),
    ("ESCAP",             "https://www.unescap.org/our-work/transport"),
]




# ── Additional seed sources merged in from FreightQuote_AI_RAG_Pipeline.ipynb (new items only, de-duplicated against FREIGHT_SEEDS above) ──
FREIGHT_SEEDS.extend([
    ("PDF Source", "https://assets.publishing.service.gov.uk/government/uploads/system/uploads/attachment_data/file/1092198/trade-remedies-guidance.pdf"),
    ("PDF Source", "https://iccwbo.org/wp-content/uploads/sites/3/2019/01/icc-incoterms-2010-publication.pdf"),
    ("PDF Source", "https://openknowledge.worldbank.org/bitstream/handle/10986/27509/114522.pdf"),
    ("PDF Source", "https://openknowledge.worldbank.org/bitstream/handle/10986/29971/120385.pdf"),
    ("PDF Source", "https://openknowledge.worldbank.org/bitstream/handle/10986/32436/9781464814358.pdf"),
    ("PDF Source", "https://openknowledge.worldbank.org/bitstream/handle/10986/33596/9781464815096.pdf"),
    ("PDF Source", "https://openknowledge.worldbank.org/bitstream/handle/10986/35016/9781464816123.pdf"),
    ("PDF Source", "https://openknowledge.worldbank.org/bitstream/handle/10986/36613/9781464816109.pdf"),
    ("PDF Source", "https://taxation-customs.ec.europa.eu/system/files/2021-12/ucc_work_programme_2021.pdf"),
    ("Web Source", "https://theloadstar.com"),
    ("PDF Source", "https://unctad.org/system/files/official-document/ditctncd2020d3_en.pdf"),
    ("PDF Source", "https://unctad.org/system/files/official-document/rmt2015_en.pdf"),
    ("PDF Source", "https://unctad.org/system/files/official-document/rmt2016_en.pdf"),
    ("PDF Source", "https://unctad.org/system/files/official-document/rmt2017_en.pdf"),
    ("PDF Source", "https://unctad.org/system/files/official-document/rmt2018_en.pdf"),
    ("PDF Source", "https://unctad.org/system/files/official-document/rmt2019_en.pdf"),
    ("PDF Source", "https://unctad.org/system/files/official-document/rmt2020_en.pdf"),
    ("PDF Source", "https://unctad.org/system/files/official-document/rmt2021_en.pdf"),
    ("PDF Source", "https://unctad.org/system/files/official-document/rmt2022_en.pdf"),
    ("PDF Source", "https://unctad.org/system/files/official-document/rmt2023_en.pdf"),
    ("PDF Source", "https://unctad.org/system/files/official-document/rmt2024_en.pdf"),
    ("PDF Source", "https://unctad.org/system/files/official-document/tdr2018_en.pdf"),
    ("PDF Source", "https://unctad.org/system/files/official-document/tdr2019_en.pdf"),
    ("PDF Source", "https://unctad.org/system/files/official-document/tdr2020_en.pdf"),
    ("PDF Source", "https://unctad.org/system/files/official-document/tdr2021_en.pdf"),
    ("PDF Source", "https://unctad.org/system/files/official-document/tdr2022_en.pdf"),
    ("PDF Source", "https://unctad.org/system/files/official-document/unctaddtlktcd2018_en.pdf"),
    ("PDF Source", "https://unctad.org/system/files/official-document/unctaddtlktcd20192_en.pdf"),
    ("PDF Source", "https://www.abf.gov.au/importing-exporting-and-manufacturing/importing/pdf/cargo-management-modernisation-guide.pdf"),
    ("PDF Source", "https://www.adb.org/sites/default/files/publication/761236/trade-finance-gaps-growth-jobs-survey-2021.pdf"),
    ("PDF Source", "https://www.apec.org/docs/default-source/publications/2022/11/2022-apec-economic-policy-report/222_ec_2022-apec-economic-policy-report.pdf"),
    ("Web Source", "https://www.balticexchange.com"),
    ("PDF Source", "https://www.bimco.org/~/media/primary-toolbar/about/publications/shipping-market-review/2022/shipping-market-review-may-2022.pdf"),
    ("Web Source", "https://www.bis.doc.gov/index.php/documents/regulation-docs/2339-ear-part-730/file"),
    ("Web Source", "https://www.cbic.gov.in"),
    ("PDF Source", "https://www.cbic.gov.in/resources//htdocs-cbec/customs/cs-act/cs-act-idx.pdf"),
    ("PDF Source", "https://www.cbp.gov/sites/default/files/assets/documents/2016-Apr/icp_trade_pubs_0.pdf"),
    ("PDF Source", "https://www.cbp.gov/sites/default/files/assets/documents/2020-Jan/Importing-into-the-United-States.pdf"),
    ("Web Source", "https://www.cbp.gov/trade"),
    ("PDF Source", "https://www.clarksons.com/media/3126/clarksons-research-shipping-review-and-outlook-spring-2022.pdf"),
    ("Web Source", "https://www.container-news.com"),
    ("PDF Source", "https://www.customs.gov.sg/files/businesses/scsb2020.pdf"),
    ("Web Source", "https://www.dgft.gov.in"),
    ("PDF Source", "https://www.dgft.gov.in/CP/FTP%202023.pdf"),
    ("PDF Source", "https://www.dnv.com/binaries/content/assets/dnv/pdfs/publications/maritime-forecast-to-2050.pdf"),
    ("Web Source", "https://www.dnv.com/maritime"),
    ("Web Source", "https://www.drewry.co.uk"),
    ("Web Source", "https://www.export.gov"),
    ("PDF Source", "https://www.fmc.gov/wp-content/uploads/2020/08/FMC-Shippers-Guide.pdf"),
    ("Web Source", "https://www.freightos.com/freight-resources"),
    ("PDF Source", "https://www.freightos.com/wp-content/uploads/2022/01/Global-Freight-Market-Report.pdf"),
    ("Web Source", "https://www.hellenicshippingnews.com"),
    ("PDF Source", "https://www.iata.org/contentassets/b6bc3e7b48cd4f44b7efba8ab1e4b4dc/dg-form-templates.pdf"),
    ("Web Source", "https://www.iata.org/en/programs/cargo/e-freight"),
    ("Web Source", "https://www.iccwbo.org/news-publications/policies-reports"),
    ("PDF Source", "https://www.ilo.org/wcmsp5/groups/public/---ed_norm/---normes/documents/publication/wcms_087817.pdf"),
    ("PDF Source", "https://www.ilo.org/wcmsp5/groups/public/---ed_protect/---protrav/---travail/documents/publication/wcms_712957.pdf"),
    ("Web Source", "https://www.imf.org/en/Publications/WEO"),
    ("Web Source", "https://www.imf.org/~/media/Files/Publications/WEO/2022/April/English/text.ashx"),
    ("Web Source", "https://www.imo.org/en/About/Conventions/Pages/Home.aspx"),
    ("Web Source", "https://www.imo.org/en/MediaCentre/HotTopics/Pages/Default.aspx"),
    ("PDF Source", "https://www.imo.org/en/OurWork/Safety/Documents/ISM%20Code%202014.pdf"),
    ("PDF Source", "https://www.intercargo.org/wp-content/uploads/2022/10/Intercargo-Annual-Report-2022.pdf"),
    ("PDF Source", "https://www.intracen.org/uploadedFiles/intracenorg/Content/Publications/The-State-of-Sustainable-Markets-Statistics-and-Emerging-Trends-2022.pdf"),
    ("Web Source", "https://www.iso.org/committee/45322.html"),
    ("Web Source", "https://www.iso.org/iso-9001-quality-management.html"),
    ("PDF Source", "https://www.lloyds.com/news-and-insights/risk-reports/library/technology-risk/global-supply-chain.pdf"),
    ("PDF Source", "https://www.marad.dot.gov/wp-content/uploads/pdf/MARAD_2021_USWaterborne_Full_Report.pdf"),
    ("Web Source", "https://www.marinetraffic.com"),
    ("PDF Source", "https://www.oecd-ilibrary.org/deliver/transport-outlook-2023_b6cc9ad6-en.pdf"),
    ("PDF Source", "https://www.oecd.org/g20/topics/trade-and-investment/G20-trade-and-growth.pdf"),
    ("PDF Source", "https://www.oecd.org/trade/oecd-wto-aid-for-trade-at-a-glance-2019-8756cfc6-en.pdf"),
    ("Web Source", "https://www.porttechnology.org"),
    ("Web Source", "https://www.spglobal.com/commodityinsights/en/market-insights/topics/shipping"),
    ("Web Source", "https://www.state.gov/trade-and-economic-issues"),
    ("Web Source", "https://www.trade.gov/harmonized-system-hs-codes"),
    ("Web Source", "https://www.trade.gov/know-your-customer"),
    ("PDF Source", "https://www.trade.gov/sites/default/files/2021-07/FTZ-Benefits-Manual.pdf"),
    ("PDF Source", "https://www.wcoomd.org/-/media/wco/public/global/pdf/topics/facilitation/instruments-and-tools/declarations/kyoto-convention/kyoto_conv.pdf"),
    ("PDF Source", "https://www.wcoomd.org/-/media/wco/public/global/pdf/topics/nomenclature/instruments-and-tools/hs-nomenclature-2022/hs-2022-edition-explanatory-notes.pdf"),
    ("Web Source", "https://www.wcoomd.org/en/topics/facilitation/instrument-and-tools/tools/single-window.aspx"),
    ("Web Source", "https://www.wto.org/english/res_e/publications_e/wtr22_e.htm"),
    ("PDF Source", "https://www.wto.org/english/res_e/reser_e/ersd202111_e.pdf"),
    ("PDF Source", "https://www.wto.org/english/res_e/statis_e/wts2022_e/wts2022_e.pdf"),
])

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def is_pdf_url(url):
    low = url.lower()
    return (low.endswith(".pdf")
            or "/pdf/" in low
            or "download=pdf" in low
            or ".pdf?" in low
            or "format=pdf" in low)


def same_domain(url, base):
    try:
        return urllib.parse.urlparse(url).netloc == urllib.parse.urlparse(base).netloc
    except Exception:
        return False


def title_from_url(url):
    name = url.rstrip("/").split("/")[-1]
    # strip extension safely without regex escape warning
    for ext in [".pdf", ".PDF", ".htm", ".html", ".aspx", ".php"]:
        if name.endswith(ext):
            name = name[: -len(ext)]
    return name.replace("-", " ").replace("_", " ").strip()[:100]


def discover_pdfs(url, category, visited=None, found=None,
                  max_depth=MAX_CRAWL_DEPTH, max_per_seed=MAX_PDFS_PER_SEED):
    if visited is None:
        visited = set()
    if found is None:
        found = []

    # ArXiv special handling
    if "arxiv.org/search" in url:
        return _crawl_arxiv(url, category, max_per_seed)

    queue = deque([(url, 0)])
    visited.add(url)

    while queue and len(found) < max_per_seed:
        cur_url, depth = queue.popleft()
        try:
            r = requests.get(cur_url, headers=HEADERS,
                             timeout=REQUEST_TIMEOUT, allow_redirects=True)
            if r.status_code != 200:
                continue
            ctype = r.headers.get("content-type", "")

            if "pdf" in ctype.lower():
                if cur_url not in [f["url"] for f in found]:
                    found.append({"url": cur_url, "category": category,
                                  "title": title_from_url(cur_url)})
                continue

            soup = BeautifulSoup(r.text, "lxml")

            for tag in soup.find_all("a", href=True):
                href = tag["href"].strip()
                abs_href = urllib.parse.urljoin(cur_url, href)
                if is_pdf_url(abs_href) and abs_href not in [f["url"] for f in found]:
                    title = (tag.get_text(strip=True) or title_from_url(abs_href))[:120]
                    found.append({"url": abs_href, "category": category, "title": title})
                    if len(found) >= max_per_seed:
                        break

            if depth < max_depth:
                for tag in soup.find_all("a", href=True):
                    href = tag["href"].strip()
                    abs_href = urllib.parse.urljoin(cur_url, href)
                    if (abs_href not in visited
                            and same_domain(abs_href, url)
                            and not is_pdf_url(abs_href)
                            and abs_href.startswith("http")
                            and "#" not in abs_href
                            and len(visited) < 200):
                        visited.add(abs_href)
                        queue.append((abs_href, depth + 1))

        except Exception:
            pass
        time.sleep(DELAY_BETWEEN)

    return found


def _crawl_arxiv(search_url, category, limit=30):
    found = []
    try:
        r = requests.get(search_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        soup = BeautifulSoup(r.text, "lxml")
        for li in soup.select("li.arxiv-result")[:limit]:
            title_tag = li.select_one("p.title")
            pdf_link  = li.select_one('a[href*="/pdf/"]')
            if pdf_link:
                pdf_url = pdf_link["href"]
                if "arxiv.org" not in pdf_url:
                    pdf_url = "https://arxiv.org" + pdf_url
                if not pdf_url.endswith(".pdf"):
                    pdf_url += ".pdf"
                title = title_tag.get_text(strip=True)[:120] if title_tag else title_from_url(pdf_url)
                found.append({"url": pdf_url, "category": category, "title": title})
    except Exception:
        pass
    return found


def download_pdf(entry, out_dir):
    url   = entry["url"]
    title = entry["title"]
    # safe filename — no regex needed
    safe  = "".join(c if c.isalnum() or c in " _-" else "_"
                    for c in f"{entry['category']}___{title}")[:100]
    fname = safe + ".pdf"
    fpath = out_dir / fname

    if fpath.exists() and fpath.stat().st_size > 1024:
        return True, fpath, "cached"

    try:
        r = requests.get(url, headers=PDF_HEADERS,
                         timeout=DOWNLOAD_TIMEOUT, allow_redirects=True, stream=True)
        ctype = r.headers.get("content-type", "")
        if r.status_code == 200:
            content = b"".join(r.iter_content(8192))
            fpath.write_bytes(content)
            kind = "pdf" if "pdf" in ctype.lower() else "html"
            return True, fpath, kind
        return False, fpath, "http_" + str(r.status_code)
    except Exception as e:
        return False, fpath, "err_" + str(e)[:40]


# ─────────────────────────────────────────────────────────────────────────────
# TEXT EXTRACTION + CHUNKING
# ─────────────────────────────────────────────────────────────────────────────

def extract_text(fpath):
    try:
        import fitz
        doc  = fitz.open(str(fpath))
        text = "\n".join(p.get_text() for p in doc)
        doc.close()
        return text
    except Exception:
        pass
    try:
        raw  = fpath.read_bytes()
        soup = BeautifulSoup(raw, "lxml")
        return soup.get_text(separator="\n", strip=True)
    except Exception:
        return ""


def chunk_text(text, title, source_url,
               chunk_words=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    words  = text.split()
    step   = chunk_words - overlap
    chunks = []
    for i in range(0, max(1, len(words) - overlap), step):
        piece = " ".join(words[i: i + chunk_words])
        if len(piece) > 80:
            cid = hashlib.md5((source_url + str(i)).encode()).hexdigest()[:12]
            chunks.append({
                "chunk_id": cid,
                "text":     piece,
                "title":    title,
                "source":   source_url,
                "offset":   i,
            })
    return chunks


# ─────────────────────────────────────────────────────────────────────────────
# RAG INDEX BUILD  (matches existing RAG notebook format exactly)
# ─────────────────────────────────────────────────────────────────────────────

def build_rag_index(chunks, out_base, project_key):
    import numpy as np
    import faiss
    from sentence_transformers import SentenceTransformer
    from rank_bm25 import BM25Okapi

    faiss_dir = out_base / "faiss_indexes"
    bm25_dir  = out_base / "bm25_indexes"
    faiss_dir.mkdir(parents=True, exist_ok=True)
    bm25_dir.mkdir(parents=True, exist_ok=True)

    ST_CACHE = "/content/.cache/sentence_transformers"
    print("  Loading sentence-transformer (all-MiniLM-L6-v2)...")
    model = SentenceTransformer("all-MiniLM-L6-v2", cache_folder=ST_CACHE)

    texts = [c["text"] for c in chunks]
    print(f"  Encoding {len(texts)} chunks...")
    embs  = model.encode(texts, show_progress_bar=True,
                         batch_size=64, convert_to_numpy=True).astype("float32")

    faiss.normalize_L2(embs)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)

    idx_path  = str(faiss_dir / (project_key + "_faiss.index"))
    meta_path = str(faiss_dir / (project_key + "_chunks_meta.json"))
    bm25_path = str(bm25_dir  / (project_key + "_bm25.pkl"))

    faiss.write_index(index, idx_path)
    with open(meta_path, "w") as f:
        json.dump(chunks, f)

    tokenized = [c["text"].lower().split() for c in chunks]
    bm25      = BM25Okapi(tokenized)
    with open(bm25_path, "wb") as f:
        pickle.dump({"bm25": bm25, "chunks": chunks}, f)

    print("  FAISS index:  " + idx_path)
    print("  Chunk meta:   " + meta_path)
    print("  BM25 index:   " + bm25_path)
    return index.ntotal


# ─────────────────────────────────────────────────────────────────────────────
# MAIN RUNNER
# ─────────────────────────────────────────────────────────────────────────────

def run_project(project_name, project_key, seeds, out_base_str):
    out_base = Path(out_base_str)
    pdf_dir  = out_base / "rag_pdfs"
    pdf_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 65)
    print("  PROJECT: " + project_name)
    print("  Seed sites: " + str(len(seeds)))
    print("  Output: " + str(out_base))
    print("=" * 65)

    # Step 1 — crawl
    print("\n  [1/4] Crawling " + str(len(seeds)) + " seed sites for PDFs...")
    all_found  = []
    visited_gl = set()
    for cat, seed_url in tqdm(seeds, desc="  Crawling"):
        found = discover_pdfs(seed_url, cat, visited=visited_gl)
        all_found.extend(found)
        print("    " + cat[:20].ljust(22) + "| " + seed_url[:50] +
              " -> " + str(len(found)) + " PDFs")

    # Deduplicate
    seen, unique = set(), []
    for e in all_found:
        if e["url"] not in seen:
            seen.add(e["url"])
            unique.append(e)
    print("\n  Unique PDFs discovered: " + str(len(unique)))

    # Step 2 — download
    print("\n  [2/4] Downloading " + str(len(unique)) + " PDFs...")
    manifest = {
        "project": project_name,
        "generated_at": datetime.now().isoformat(),
        "total_discovered": len(unique),
        "downloaded": 0, "cached": 0, "failed": 0,
        "files": [],
    }
    for entry in tqdm(unique, desc="  Downloading"):
        ok, fpath, status = download_pdf(entry, pdf_dir)
        entry["local_path"] = str(fpath)
        entry["status"]     = status
        if ok:
            if status == "cached":
                manifest["cached"] += 1
            else:
                manifest["downloaded"] += 1
        else:
            manifest["failed"] += 1
        manifest["files"].append(entry)
        time.sleep(DELAY_BETWEEN)

    with open(out_base / "manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)

    print("  Downloaded: " + str(manifest["downloaded"]) +
          " | Cached: " + str(manifest["cached"]) +
          " | Failed: " + str(manifest["failed"]))

    # Step 3 — chunk
    print("\n  [3/4] Extracting text and chunking...")
    all_chunks = []
    ok_files   = [e for e in manifest["files"]
                  if e.get("status") in ("pdf", "html", "cached")]
    for entry in tqdm(ok_files, desc="  Chunking"):
        fpath = Path(entry.get("local_path", ""))
        if not fpath.exists() or fpath.stat().st_size < 512:
            continue
        text = extract_text(fpath)
        if len(text.split()) < 20:
            continue
        all_chunks.extend(chunk_text(text, entry["title"], entry["url"]))
    print("  Total chunks: " + str(len(all_chunks)))

    # Step 4 — index
    print("\n  [4/4] Building FAISS + BM25 indexes...")
    n = 0
    if all_chunks:
        n = build_rag_index(all_chunks, out_base, project_key)
    else:
        print("  No chunks to index")
    print("\n  " + project_name + " COMPLETE: " + str(n) + " vectors")
    return manifest


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    DRIVE = Path("/content/drive/MyDrive")

    if DRIVE.exists():

        FQ_BASE = str(DRIVE / FQ_DRIVE_FOLDER)
        print("Google Drive mounted.")

        print("FreightQuote -> " + FQ_BASE)
    else:

        FQ_BASE = "/content/FreightQuote_AI"
        print("Drive not mounted — saving locally.")


    print("FreightQuote seed sites: " + str(len(FREIGHT_SEEDS)))


    fq_m = run_project("FreightQuote AI", "freight",   FREIGHT_SEEDS,   FQ_BASE)

    print("\n" + "=" * 65)
    print("  FINAL SUMMARY")
    print("=" * 65)
    for nm, m in [("FreightQuote", fq_m)]:
        print("  " + nm.ljust(14) +
              " | Found: " + str(m["total_discovered"]) +
              " | DL: " + str(m["downloaded"]) +
              " | Cached: " + str(m["cached"]) +
              " | Failed: " + str(m["failed"]))
    print("=" * 65)


In [ ]:
!pip install -q requests beautifulsoup4 lxml tqdm PyMuPDF sentence-transformers faiss-cpu rank-bm25
!python3 /content/pdf_web_crawler.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 52.0 MB/s eta 0:00:00
Drive not mounted — saving locally.
FreightQuote seed sites: 101

  PROJECT: FreightQuote AI
  Seed sites: 101
  Output: /content/FreightQuote_AI

  [1/4] Crawling 101 seed sites for PDFs...
  Crawling:   0% 0/101 [00:00<?, ?it/s]    Web Source            | https://www.iata.org/en/programs/cargo/security -> 0 PDFs
  Crawling:   1% 1/101 [00:00<01:00,  1.64it/s]    Web Source            | https://www.iata.org/en/programs/cargo/live-animal -> 50 PDFs
  Crawling:   2% 2/101 [00:33<32:10, 19.50s/it]    Web Source            | https://www.classnk.or.jp/hp/en/index.html -> 4 PDFs
  Crawling:   3% 3/101 [00:34<18:00, 11.03s/it]    Web Source            | https://www.trade.gov/trade-data-and-analysis -> 1 PDFs
  Crawling:   4% 4/101 [00:34<11:06,  6.87s/it]    Web Source            | http://www.wcoomd.org/en/topics/nomenclature/overv -> 1

# FreightQuote RAG Builder
# Storage: FAISS/BM25/ML models -> Google Drive | LLM -> Local only


In [ ]:
!pip install -q faiss-cpu sentence-transformers rank-bm25 pymupdf
!pip install -q langchain langchain-community langchain-text-splitters
!pip install -q plotly pandas numpy scikit-learn joblib
print('All FreightQuote RAG dependencies installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
All FreightQuote RAG dependencies installed


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

DRIVE_BASE = '/content/drive/MyDrive/FreightQuote_AI'
FAISS_DIR  = f'{DRIVE_BASE}/faiss_indexes'
BM25_DIR   = f'{DRIVE_BASE}/bm25_indexes'
ML_DIR     = f'{DRIVE_BASE}/ml_models'
PDF_DIR    = f'{DRIVE_BASE}/rag_pdfs'
KB_DIR     = f'{DRIVE_BASE}/synthetic_kb'
DB_PATH    = f'{DRIVE_BASE}/freight_database.db'
LLM_CACHE  = '/content/.cache/hf_models'
ST_CACHE   = '/content/.cache/sentence_transformers'

for d in [FAISS_DIR, BM25_DIR, ML_DIR, PDF_DIR, KB_DIR, LLM_CACHE, ST_CACHE]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted. Storage:')
print(f'  FAISS  -> {FAISS_DIR}')
print(f'  BM25   -> {BM25_DIR}')
print(f'  ML     -> {ML_DIR}')
print(f'  LLM    -> {LLM_CACHE} (local, not Drive)')


Mounted at /content/drive
Drive mounted. Storage:
  FAISS  -> /content/drive/MyDrive/FreightQuote_AI/faiss_indexes
  BM25   -> /content/drive/MyDrive/FreightQuote_AI/bm25_indexes
  ML     -> /content/drive/MyDrive/FreightQuote_AI/ml_models
  LLM    -> /content/.cache/hf_models (local, not Drive)


In [ ]:
import json

FREIGHT_DOCS = [
    {'title': 'INCOTERMS-EXW: Ex Works',
     'content': 'EXW means the seller makes goods available at their premises. The buyer bears all costs and risks from that point. The seller has minimal obligation - no export clearance, no loading. Used for domestic trade or when buyer has own logistics. Risk transfers when goods are placed at disposal of buyer at named place. Least responsibility for seller, maximum responsibility for buyer.'},
    {'title': 'INCOTERMS-FOB: Free on Board',
     'content': 'FOB requires seller to deliver goods on board the vessel nominated by buyer at named port of shipment. Seller handles export clearance. Risk transfers once goods are on board. Used for bulk cargo and when buyer controls freight arrangements. Seller pays for loading costs. Most commonly used INCOTERM for sea freight.'},
    {'title': 'INCOTERMS-CIF: Cost Insurance Freight',
     'content': 'CIF requires seller to arrange and pay for main carriage and insurance to named port of destination. Risk transfers when goods are on board vessel at port of origin. Seller provides minimum insurance at 110% of contract value under Institute Cargo Clauses C. Buyer pays unloading, import duties, and inland freight at destination.'},
    {'title': 'INCOTERMS-DDP: Delivered Duty Paid',
     'content': 'DDP places maximum obligation on seller who must deliver goods to named destination cleared for import with all duties paid. Seller handles export and import formalities. Highest risk and cost for seller. Buyer has least responsibility receiving goods at destination. Used when seller wants to provide door-to-door service.'},
    {'title': 'INCOTERMS-FCA: Free Carrier',
     'content': 'FCA requires seller to deliver goods to a named carrier or other nominated person at sellers premises or another named place. Used for multimodal transport including containers. Seller handles export clearance. Risk transfers when goods are handed to carrier at named location.'},
    {'title': 'INCOTERMS-DAP: Delivered at Place',
     'content': 'DAP requires seller to deliver goods ready for unloading at named destination. Seller bears all costs and risks to destination. Buyer handles import clearance and duties. Common for door-to-door shipments where seller controls logistics but buyer handles import formalities.'},
    {'title': 'HS Code Classification: Electronics Chapter 85',
     'content': 'Chapter 85 covers electrical machinery, equipment, and parts. HS code 8517 covers telephone sets including smartphones. HS 8471 covers computers and data processing units. Import duty varies 0-15% depending on country. Most electronics qualify for IT Agreement duty-free treatment in 82 countries. CE marking mandatory for EU markets. Technical compliance certificates required for US and EU.'},
    {'title': 'HS Code Classification: Textiles Chapters 50-63',
     'content': 'Textile HS codes span Chapters 50-63. Cotton fabric codes 5208-5212. Synthetic fabrics 5407-5408. Ready-made garments 6101-6117. Rules of Origin require substantial transformation typically two tariff heading changes. Anti-dumping duties may apply on certain country-product combinations. EU GSP provides preferential rates for developing country exports.'},
    {'title': 'HS Code Classification: Pharmaceuticals Chapter 30',
     'content': 'Chapter 30 covers pharmaceutical products. HS 3004 covers medicaments for retail sale. Import requires FDA US or EMA EU certification for finished products. Cold chain shipments for biologics require temperature monitoring at 2-8 degrees Celsius. Special permits required for controlled substances. Falsified medicines regulation requires track-and-trace serialization in EU.'},
    {'title': 'HS Code Classification: Dangerous Goods IMDG Code',
     'content': 'Dangerous goods are classified in 9 IMDG classes. Class 1 Explosives, Class 2 Gases, Class 3 Flammable Liquids, Class 4 Flammable Solids, Class 5 Oxidizers, Class 6 Toxic and Infectious, Class 7 Radioactive, Class 8 Corrosives, Class 9 Miscellaneous. Each class has specific packaging, labeling, and documentation requirements. Shippers Declaration of Dangerous Goods is mandatory.'},
    {'title': 'Customs Regulation: United States CBP',
     'content': 'US Customs and Border Protection enforces trade laws. All shipments above 2500 USD require formal entry with ISF submitted 24 hours before vessel loading. Anti-dumping and countervailing duties apply to specific countries and products. FDA prior notice required for food and beverage shipments. DEA permit for controlled substances. Continuous customs bonds required for frequent importers.'},
    {'title': 'Customs Regulation: European Union',
     'content': 'EU customs union applies common external tariff across 27 member states. Single Administrative Document required for all formal entries. Import One-Stop-Shop simplifies VAT for e-commerce below EUR 150. Authorized Economic Operator status provides expedited clearance benefits. REACH compliance required for chemicals. CE marking mandatory for electronics and machinery sold in EU.'},
    {'title': 'Customs Regulation: India ICEGATE',
     'content': 'Indian Customs governed by Customs Act 1962. Import duty equals Basic Customs Duty plus Social Welfare Surcharge plus IGST. Bill of Entry must be filed within 30 days of vessel arrival. Advance Authorization allows duty-free import of inputs for export production. ICEGATE is the online customs portal for e-filing. FSSAI clearance needed for food products.'},
    {'title': 'Customs Regulation: China GACC',
     'content': 'China Customs administered by GACC General Administration of Customs. H2030 reform modernized procedures with online declaration via Single Window system. Food imports require GACC-registered foreign facilities. Cross-border e-commerce under 2000 RMB eligible for simplified clearance. China applies MFN, Preferential, and Special tariff rates. RCEP provides preferential rates with 15 Asia-Pacific partners.'},
    {'title': 'Customs Regulation: Middle East UAE and Saudi Arabia',
     'content': 'UAE Customs aligned with GCC Common Customs Law. 5% unified customs duty applies to most goods with some exemptions. Import declarations via Dubai Trade or Abu Dhabi ADCC portals. Halal certification mandatory for food products. Saudi Arabia requires SASO product certification for electronics and toys. Both countries impose 15% VAT on imported goods.'},
    {'title': 'Port Operations: Singapore PSA',
     'content': 'Singapore PSA handles 37 million TEUs annually. Tuas Mega Port will be worlds largest fully automated port. Standard vessel turnaround 24-48 hours. Congestion Index typically 0.2-0.4 which is Low. Transit hub for Southeast Asia with 200 plus shipping lines connected to 600 plus ports. PORTNET digital system integrates all stakeholders. Container tracking available real-time.'},
    {'title': 'Port Operations: Shanghai SIPG',
     'content': 'Shanghai is worlds busiest container port at 47 million TEUs in 2023. Yangshan Deep Water Port serves mega-vessels with 24000 TEU capacity. Average vessel dwell time 3-4 days. Congestion peaks during Chinese New Year and Golden Week. CNPP platform enables e-filing. Port congestion surcharges applied during peak seasons typically USD 200-500 per TEU.'},
    {'title': 'Port Operations: Rotterdam Europe Largest',
     'content': 'Rotterdam is Europes largest port handling 450 million tons of cargo annually. 40km port complex with deep water draft of 24m. Average container dwell 4-6 days. Maasvlakte 2 automated terminal handles 4.5 million TEU. Rotterdam Port Community System integrates 7500 companies. 24/7 customs clearance available. Connected to European hinterland by 325 inland barges daily.'},
    {'title': 'Port Operations: Dubai Jebel Ali DP World',
     'content': 'Jebel Ali is worlds 9th largest port and largest in Middle East at 14.5 million TEU. 67 berths with 22m draft for ultra-large vessels. Average dwell 5-7 days. Jebel Ali Free Zone JAFZA provides 0% import duty for re-exports. Customs clearance via Mirsal2 system. 24/7 operations with automated stacking cranes. Transit hub for Africa South Asia and CIS countries.'},
    {'title': 'Freight Pricing: FCL vs LCL',
     'content': 'Full Container Load FCL means shipper pays for entire container regardless of cargo volume. 20ft container capacity 25-28 CBM and 18-22 tons. 40ft container 55-67 CBM and 26-27 tons. 40ft High Cube 72-76 CBM. FCL preferred when cargo exceeds 15 CBM or is security-sensitive. Less-than-Container-Load LCL consolidates cargo. Rate quoted per CBM or per ton whichever is greater.'},
    {'title': 'Freight Pricing: Surcharges and Add-ons',
     'content': 'Standard freight rate includes base ocean freight only. Additional surcharges include Bunker Adjustment Factor BAF for fuel cost volatility at USD 50-400 per TEU. Terminal Handling Charges THC at origin and destination USD 100-300 per TEU. Bill of Lading fee USD 50-75. Peak Season Surcharge PSS in Q3-Q4 at USD 200-800 per TEU. Congestion Surcharge when port utilization above 90%.'},
    {'title': 'Freight Pricing: Dynamic Pricing Model',
     'content': 'Modern freight pricing uses algorithmic dynamic pricing. Key variables are current capacity utilization which is most impactful, fuel price Brent crude, port congestion index, demand forecast including seasonality, exchange rates, and competitor rates. Machine learning models predict rate movements 2-4 weeks forward with 78% accuracy. Spot rates volatile so book 3-4 weeks in advance for savings of 15-25%.'},
    {'title': 'Freight Pricing: Insurance Calculation',
     'content': 'Cargo insurance premium equals Insured Value multiplied by Rate. Standard rate 0.2-0.8% of CIF value depending on commodity route and packaging. All-Risk policy covers all physical loss or damage. Institute Cargo Clauses A, B, C provide different levels of coverage. War risk insurance separate at 0.01-0.25% depending on route. High-value electronics attract 0.5-1.2% premium rate.'},
    {'title': 'Freight Pricing: CO2 Emission Calculation',
     'content': 'Maritime CO2 emissions calculated per tonne-kilometer. Container ship emission factor is 16-24 grams CO2 per tonne-km depending on vessel size and speed. Total Emission equals Weight in tonnes multiplied by Distance in km multiplied by Emission Factor divided by 1000000. IMO 2023 regulations require 40% carbon intensity reduction by 2030. Carbon offset cost USD 15-45 per tonne CO2.'},
    {'title': 'Carrier Performance: Top Container Lines',
     'content': 'Top 10 container lines control 85% of global capacity. Rankings by TEU capacity: MSC 5.2 million, Maersk 4.3 million, CMA CGM 3.6 million, COSCO 3.1 million, Hapag-Lloyd 2.1 million, ONE 1.5 million, Evergreen 1.6 million, HMM 0.8 million, Yang Ming 0.7 million, PIL 0.35 million TEU. Alliance membership affects route coverage through 2M Alliance Ocean Alliance and THE Alliance.'},
    {'title': 'Carrier Safety and Reliability Standards',
     'content': 'Carrier safety assessed by P&I Club insurance coverage, safety management system ISM Code certification, vessel age preferably under 15 years, SOLAS compliance, annual safety inspection records, on-time performance where industry average is 50-65% and best carriers achieve 75% plus, and damage rate with target below 0.5% of shipments. Class society approval from Lloyds Register Bureau Veritas DNV or ABS.'},
    {'title': 'Carrier Selection Criteria',
     'content': 'Key criteria for carrier selection include schedule reliability with published on-time departure and arrival rates, port coverage and transit time, equipment availability including container supply, pricing covering spot versus contract options, documentation quality, claims settlement history, digital tools including API integration and track-and-trace, and sustainability credentials. Drewry and BlueWater Reporting publish quarterly carrier performance scorecards.'},
    {'title': 'Route Risk Assessment Framework',
     'content': 'Route risk score is calculated from weather severity weight 30%, port congestion weight 25%, geopolitical risk weight 20%, piracy risk weight 15%, and infrastructure quality weight 10%. High risk routes score above 7 out of 10. Recommended routes have risk score below 4 and reliability index above 0.8. Transit time variance is key metric with target below 15% standard deviation from published schedule.'},
    {'title': 'Margin Optimization Strategies',
     'content': 'Freight margin optimization focuses on five levers. Dynamic pricing based on capacity utilization captures 8-12% margin improvement. Fuel hedging through forward contracts reduces cost volatility by 30%. Volume commitments with carriers provide 10-20% rate discount. Port selection optimization to avoid congestion surcharges saves USD 200-600 per TEU. Insurance bundling for high-volume customers reduces premium by 15%.'},
]

# ── Additional documents merged in from FreightQuote_AI_RAG_Pipeline.ipynb (new items only, de-duplicated against FREIGHT_DOCS above) ──
FREIGHT_DOCS.extend([
    {'title': 'Rule CUST-127',
     'content': 'CFR (Cost and Freight) is identical to CIF except the seller is not required to procure marine insurance.'},
    {'title': 'Rule CUST-130',
     'content': 'CPT (Carriage Paid To) requires the seller to pay for carriage to the named destination, with risk transferring once goods are handed to the first carrier.'},
    {'title': 'Rule CUST-131',
     'content': 'A Free Trade Zone (FTZ) allows goods to be imported, stored, manufactured, or re-exported without formal customs duties until they enter the domestic market.'},
    {'title': 'Rule CUST-132',
     'content': 'A Free Trade Agreement (FTA) reduces or eliminates tariffs between member countries on qualifying goods that meet rules-of-origin criteria.'},
    {'title': 'Rule CUST-133',
     'content': 'Dangerous Goods (DG) cargo must be classified into one of nine IMO/IATA hazard classes before it can be booked for transport.'},
    {'title': 'Rule CUST-134',
     'content': 'A Dangerous Goods Declaration (DGD) must accompany every shipment of hazardous cargo and match the packaging, labeling, and UN number.'},
    {'title': 'Rule CUST-135',
     'content': 'Reefer (refrigerated) containers require continuous temperature monitoring and a genset connection at every transshipment point.'},
    {'title': 'Rule CUST-136',
     'content': 'A Straight Bill of Lading is non-negotiable and consigns cargo to a named party only, unlike an Order Bill of Lading which can be transferred by endorsement.'},
    {'title': 'Rule CUST-137',
     'content': 'A Telex Release allows a shipper to release an original Bill of Lading electronically so the consignee can collect cargo without the paper original.'},
    {'title': 'Rule CUST-138',
     'content': 'Free time at a port is the number of days a container may sit at the terminal or with the consignee before demurrage or detention charges begin.'},
    {'title': 'Rule CUST-141',
     'content': 'A House Bill of Lading (HBL) is issued by a freight forwarder to the shipper, while a Master Bill of Lading (MBL) is issued by the ocean carrier to the forwarder.'},
    {'title': 'Rule CUST-142',
     'content': 'Chargeable weight for air freight is the greater of actual gross weight or volumetric weight, where volumetric weight (kg) = L x W x H (cm) / 6000.'},
    {'title': 'Rule CUST-143',
     'content': 'A Certificate of Analysis (CoA) is often required for chemical, pharmaceutical, and food shipments to confirm product composition and quality.'},
    {'title': 'Rule CUST-144',
     'content': 'Know Your Customer (KYC) screening against denied-party and sanctions lists is mandatory before onboarding a new shipper or consignee.'},
    {'title': 'Rule CUST-145',
     'content': 'Duty drawback allows an importer to reclaim duties paid on imported goods that are subsequently re-exported or used in export manufacturing.'},
    {'title': 'Rule CUST-146',
     'content': 'A bonded warehouse allows imported goods to be stored without payment of duty until the goods are withdrawn for domestic consumption.'},
    {'title': 'Rule CUST-147',
     'content': 'The World Customs Organization SAFE Framework promotes Authorized Economic Operator (AEO) status for trusted traders, enabling expedited clearance.'},
    {'title': 'Rule CUST-148',
     'content': 'Container Freight Station (CFS) charges cover the cost of stuffing and destuffing LCL cargo at the terminal.'},
    {'title': 'Rule CUST-149',
     'content': "A Shipper's Letter of Instruction (SLI) provides the freight forwarder with shipment details needed to prepare export documentation."},
    {'title': 'Rule CUST-150',
     'content': 'Rules of origin determine which country a product is considered to originate from for tariff and trade-agreement purposes.'},
    {'title': 'Rule CUST-151',
     'content': 'A pre-shipment inspection (PSI) verifies quality, quantity, and conformity of goods before they are shipped, often required by the importing country.'},
    {'title': 'Rule CUST-152',
     'content': 'The Incoterms rules are published and updated by the International Chamber of Commerce (ICC), with the current edition being Incoterms 2020.'},
    {'title': 'Rule CUST-153',
     'content': 'General Average is a maritime law principle where all parties in a sea venture proportionally share losses from a voluntary sacrifice made to save the voyage.'},
    {'title': 'Rule CUST-154',
     'content': 'A Notify Party listed on a Bill of Lading is informed of cargo arrival but is not the owner or consignee of the goods.'},
    {'title': 'Rule CUST-155',
     'content': 'SOLAS VGM (Verified Gross Mass) regulations require shippers to declare and verify the gross mass of a packed container before it can be loaded on a vessel.'},
    {'title': 'Rule CUST-156',
     'content': "A freight forwarder's Non-Vessel-Operating Common Carrier (NVOCC) license allows it to issue its own Bills of Lading without owning vessels."},
    {'title': 'Rule CUST-157',
     'content': 'Trade compliance screening should flag any shipment where the declared value deviates more than 25% from the ML-predicted benchmark cost for review.'},
    {'title': 'Rule CUST-158',
     'content': 'Carrier tier ratings in FreightQuote AI are classified as Tier 1 (Apex), Tier 2 (Standard), or Tier 3 (Watchlist) based on punctuality rate and tariff compliance score.'},
    {'title': 'Rule CUST-159',
     'content': 'A carrier is auto-flagged for audit in FreightQuote AI when its punctuality rate falls below 0.85 or its average delay exceeds 3 days.'},
    {'title': 'Rule CUST-160',
     'content': 'Route Optimization Agent recommendations should consider port congestion status, average wind speed, and historical delay penalty multiplier before proposing an alternate route.'},
    {'title': 'Country Customs Profile: United States',
     'content': 'Customs profile — United States: the primary customs authority is U.S. Customs and Border Protection (CBP). The typical import tax/VAT/GST rate is no federal VAT; state sales tax varies. Standard import documentation includes: Entry Summary (CBP Form 7501), ISF 10+2, Commercial Invoice.'},
    {'title': 'Country Customs Profile: United Kingdom',
     'content': 'Customs profile — United Kingdom: the primary customs authority is HM Revenue & Customs (HMRC). The typical import tax/VAT/GST rate is 20% standard VAT. Standard import documentation includes: Customs Declaration Service (CDS) entry, Commercial Invoice, EORI number.'},
    {'title': 'Country Customs Profile: Germany',
     'content': 'Customs profile — Germany: the primary customs authority is German Customs (Zoll), under the EU Union Customs Code. The typical import tax/VAT/GST rate is 19% standard VAT. Standard import documentation includes: Single Administrative Document (SAD), EORI number, Commercial Invoice.'},
    {'title': 'Country Customs Profile: France',
     'content': 'Customs profile — France: the primary customs authority is Direction Generale des Douanes (DGDDI). The typical import tax/VAT/GST rate is 20% standard VAT. Standard import documentation includes: Single Administrative Document (SAD), EORI number.'},
    {'title': 'Country Customs Profile: India',
     'content': 'Customs profile — India: the primary customs authority is Central Board of Indirect Taxes and Customs (CBIC). The typical import tax/VAT/GST rate is 18% standard GST (varies by HS chapter). Standard import documentation includes: Bill of Entry filed via ICEGATE, Import Export Code (IEC).'},
    {'title': 'Country Customs Profile: China',
     'content': 'Customs profile — China: the primary customs authority is General Administration of Customs (GACC). The typical import tax/VAT/GST rate is 13% standard VAT. Standard import documentation includes: Customs Declaration Form, China Compulsory Certification (CCC) where applicable.'},
    {'title': 'Country Customs Profile: Japan',
     'content': 'Customs profile — Japan: the primary customs authority is Japan Customs. The typical import tax/VAT/GST rate is 10% consumption tax. Standard import documentation includes: Import Declaration (Form C-5020), Certificate of Origin.'},
    {'title': 'Country Customs Profile: Singapore',
     'content': 'Customs profile — Singapore: the primary customs authority is Singapore Customs. The typical import tax/VAT/GST rate is 9% GST. Standard import documentation includes: Customs In-Non-Payment Permit via TradeNet, Certificate of Origin.'},
    {'title': 'Country Customs Profile: Australia',
     'content': 'Customs profile — Australia: the primary customs authority is Australian Border Force (ABF). The typical import tax/VAT/GST rate is 10% GST. Standard import documentation includes: Import Declaration (N10), Tariff Classification.'},
    {'title': 'Country Customs Profile: Canada',
     'content': 'Customs profile — Canada: the primary customs authority is Canada Border Services Agency (CBSA). The typical import tax/VAT/GST rate is 5% federal GST plus provincial tax. Standard import documentation includes: Commercial Invoice, B3 Canada Customs Coding Form.'},
    {'title': 'Country Customs Profile: Brazil',
     'content': 'Customs profile — Brazil: the primary customs authority is Receita Federal do Brasil. The typical import tax/VAT/GST rate is Import tax plus IPI and ICMS (varies by state). Standard import documentation includes: Declaracao de Importacao (DI/DUIMP), Import License (LI) where required.'},
    {'title': 'Country Customs Profile: Mexico',
     'content': 'Customs profile — Mexico: the primary customs authority is Servicio de Administracion Tributaria (SAT). The typical import tax/VAT/GST rate is 16% standard VAT. Standard import documentation includes: Pedimento de Importacion, Certificate of Origin under USMCA.'},
    {'title': 'Country Customs Profile: South Korea',
     'content': 'Customs profile — South Korea: the primary customs authority is Korea Customs Service (KCS). The typical import tax/VAT/GST rate is 10% VAT. Standard import documentation includes: Import Declaration via UNI-PASS, Certificate of Origin.'},
    {'title': 'Country Customs Profile: United Arab Emirates',
     'content': 'Customs profile — United Arab Emirates: the primary customs authority is Federal Customs Authority (FCA). The typical import tax/VAT/GST rate is 5% VAT. Standard import documentation includes: Import Declaration, Certificate of Origin, Bill of Lading.'},
    {'title': 'Country Customs Profile: Saudi Arabia',
     'content': 'Customs profile — Saudi Arabia: the primary customs authority is Zakat, Tax and Customs Authority (ZATCA). The typical import tax/VAT/GST rate is 15% VAT. Standard import documentation includes: SABER conformity certificate, Import Declaration.'},
    {'title': 'Country Customs Profile: South Africa',
     'content': 'Customs profile — South Africa: the primary customs authority is South African Revenue Service (SARS) Customs. The typical import tax/VAT/GST rate is 15% VAT. Standard import documentation includes: Customs Clearance Declaration (SAD 500), Import Permit where required.'},
    {'title': 'Country Customs Profile: Netherlands',
     'content': 'Customs profile — Netherlands: the primary customs authority is Dutch Customs, under the EU Union Customs Code. The typical import tax/VAT/GST rate is 21% standard VAT. Standard import documentation includes: Single Administrative Document (SAD), EORI number.'},
    {'title': 'Country Customs Profile: Italy',
     'content': 'Customs profile — Italy: the primary customs authority is Agenzia delle Dogane e dei Monopoli. The typical import tax/VAT/GST rate is 22% standard VAT. Standard import documentation includes: Single Administrative Document (SAD), EORI number.'},
    {'title': 'Country Customs Profile: Spain',
     'content': 'Customs profile — Spain: the primary customs authority is Agencia Tributaria - Aduanas. The typical import tax/VAT/GST rate is 21% standard VAT. Standard import documentation includes: Single Administrative Document (SAD), EORI number.'},
    {'title': 'Country Customs Profile: Vietnam',
     'content': 'Customs profile — Vietnam: the primary customs authority is Vietnam Customs (General Department of Customs). The typical import tax/VAT/GST rate is 10% standard VAT. Standard import documentation includes: Customs Declaration via VNACCS, Certificate of Origin.'},
    {'title': 'Country Customs Profile: Indonesia',
     'content': 'Customs profile — Indonesia: the primary customs authority is Directorate General of Customs and Excise (DGCE). The typical import tax/VAT/GST rate is 11% VAT. Standard import documentation includes: Import Declaration (PIB), Import Identification Number (API).'},
    {'title': 'Country Customs Profile: Thailand',
     'content': 'Customs profile — Thailand: the primary customs authority is Thai Customs Department. The typical import tax/VAT/GST rate is 7% VAT. Standard import documentation includes: Import Declaration via e-Customs, Certificate of Origin.'},
    {'title': 'Country Customs Profile: Malaysia',
     'content': 'Customs profile — Malaysia: the primary customs authority is Royal Malaysian Customs Department. The typical import tax/VAT/GST rate is 10% sales tax on select goods. Standard import documentation includes: Customs Form No. 1 (K1), Certificate of Origin.'},
    {'title': 'Country Customs Profile: Philippines',
     'content': 'Customs profile — Philippines: the primary customs authority is Bureau of Customs (BOC). The typical import tax/VAT/GST rate is 12% VAT. Standard import documentation includes: Import Entry and Internal Revenue Declaration (IEIRD).'},
    {'title': 'Country Customs Profile: Turkey',
     'content': 'Customs profile — Turkey: the primary customs authority is Turkish Customs Administration. The typical import tax/VAT/GST rate is 20% standard VAT. Standard import documentation includes: Customs Declaration (Gumruk Beyannamesi), ATR Certificate for EU trade.'},
    {'title': 'Country Customs Profile: Egypt',
     'content': 'Customs profile — Egypt: the primary customs authority is Egyptian Customs Authority. The typical import tax/VAT/GST rate is 14% VAT. Standard import documentation includes: ACID (Advance Cargo Information Declaration), Import License.'},
    {'title': 'Country Customs Profile: Nigeria',
     'content': 'Customs profile — Nigeria: the primary customs authority is Nigeria Customs Service (NCS). The typical import tax/VAT/GST rate is 7.5% VAT. Standard import documentation includes: Pre-Arrival Assessment Report (PAAR), Form M.'},
    {'title': 'Country Customs Profile: Kenya',
     'content': 'Customs profile — Kenya: the primary customs authority is Kenya Revenue Authority (KRA) Customs. The typical import tax/VAT/GST rate is 16% VAT. Standard import documentation includes: Import Declaration Form (IDF), Certificate of Conformity.'},
    {'title': 'Country Customs Profile: Poland',
     'content': 'Customs profile — Poland: the primary customs authority is Krajowa Administracja Skarbowa (KAS) Customs. The typical import tax/VAT/GST rate is 23% standard VAT. Standard import documentation includes: Single Administrative Document (SAD), EORI number.'},
    {'title': 'Country Customs Profile: Sweden',
     'content': 'Customs profile — Sweden: the primary customs authority is Swedish Customs (Tullverket). The typical import tax/VAT/GST rate is 25% standard VAT. Standard import documentation includes: Single Administrative Document (SAD), EORI number.'},
    {'title': 'Country Customs Profile: Switzerland',
     'content': 'Customs profile — Switzerland: the primary customs authority is Swiss Federal Office for Customs and Border Security (FOCBS). The typical import tax/VAT/GST rate is 8.1% standard VAT. Standard import documentation includes: Import Declaration (e-dec), Certificate of Origin.'},
    {'title': 'Country Customs Profile: New Zealand',
     'content': 'Customs profile — New Zealand: the primary customs authority is New Zealand Customs Service. The typical import tax/VAT/GST rate is 15% GST. Standard import documentation includes: Import Entry via TSW, Certificate of Origin.'},
    {'title': 'Country Customs Profile: Argentina',
     'content': 'Customs profile — Argentina: the primary customs authority is Direccion General de Aduanas (DGA). The typical import tax/VAT/GST rate is 21% VAT. Standard import documentation includes: Declaracion Jurada Anticipada de Importacion (DJAI equivalent), Import License.'},
    {'title': 'Country Customs Profile: Chile',
     'content': 'Customs profile — Chile: the primary customs authority is Servicio Nacional de Aduanas. The typical import tax/VAT/GST rate is 19% VAT. Standard import documentation includes: Import Declaration (DIN), Certificate of Origin.'},
    {'title': 'Country Customs Profile: Israel',
     'content': 'Customs profile — Israel: the primary customs authority is Israel Tax Authority Customs. The typical import tax/VAT/GST rate is 17% VAT. Standard import documentation includes: Import Declaration, Standards Institute Approval where required.'},
    {'title': 'Country Customs Profile: Bangladesh',
     'content': 'Customs profile — Bangladesh: the primary customs authority is National Board of Revenue (NBR) Customs. The typical import tax/VAT/GST rate is 15% VAT. Standard import documentation includes: Bill of Entry via ASYCUDA World, Import Registration Certificate (IRC).'},
    {'title': 'Country Customs Profile: Pakistan',
     'content': 'Customs profile — Pakistan: the primary customs authority is Pakistan Customs (FBR). The typical import tax/VAT/GST rate is 18% sales tax. Standard import documentation includes: Goods Declaration via WeBOC, Import Authorization.'},
    {'title': 'Country Customs Profile: Colombia',
     'content': 'Customs profile — Colombia: the primary customs authority is Direccion de Impuestos y Aduanas Nacionales (DIAN). The typical import tax/VAT/GST rate is 19% VAT. Standard import documentation includes: Import Declaration (Declaracion de Importacion), Certificate of Origin.'},
    {'title': 'Country Customs Profile: Ireland',
     'content': 'Customs profile — Ireland: the primary customs authority is Irish Revenue Customs. The typical import tax/VAT/GST rate is 23% standard VAT. Standard import documentation includes: Single Administrative Document (SAD), EORI number.'},
    {'title': 'Country Customs Profile: Qatar',
     'content': 'Customs profile — Qatar: the primary customs authority is General Authority of Customs. The typical import tax/VAT/GST rate is 5% VAT (select goods). Standard import documentation includes: Import Declaration, Certificate of Origin.'},
    {'title': 'Port Profile: Port of Shanghai',
     'content': 'Port profile — Port of Shanghai (China): primary mode is Ocean container. Busiest container port in the world by TEU throughput. Shipments routed through Port of Shanghai should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Port of Singapore',
     'content': 'Port profile — Port of Singapore (Singapore): primary mode is Ocean container / transshipment hub. Major transshipment hub for Southeast Asia and the Strait of Malacca. Shipments routed through Port of Singapore should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Port of Ningbo-Zhoushan',
     'content': 'Port profile — Port of Ningbo-Zhoushan (China): primary mode is Ocean container / bulk. Among the highest cargo-tonnage ports globally. Shipments routed through Port of Ningbo-Zhoushan should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Port of Shenzhen',
     'content': 'Port profile — Port of Shenzhen (China): primary mode is Ocean container. Key gateway for the Pearl River Delta manufacturing region. Shipments routed through Port of Shenzhen should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Port of Guangzhou',
     'content': 'Port profile — Port of Guangzhou (China): primary mode is Ocean container / bulk. Major hub for the Guangdong manufacturing belt. Shipments routed through Port of Guangzhou should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Port of Busan',
     'content': 'Port profile — Port of Busan (South Korea): primary mode is Ocean container / transshipment. Primary transshipment hub for Northeast Asia. Shipments routed through Port of Busan should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Port of Qingdao',
     'content': 'Port profile — Port of Qingdao (China): primary mode is Ocean container / bulk. Major northern China deep-water port. Shipments routed through Port of Qingdao should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Port of Hong Kong',
     'content': 'Port profile — Port of Hong Kong (Hong Kong SAR): primary mode is Ocean container / transshipment. Historic free port and regional logistics hub. Shipments routed through Port of Hong Kong should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Jebel Ali Port',
     'content': 'Port profile — Jebel Ali Port (United Arab Emirates): primary mode is Ocean container / free zone. Largest man-made harbor and Middle East transshipment hub. Shipments routed through Jebel Ali Port should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Port of Rotterdam',
     'content': 'Port profile — Port of Rotterdam (Netherlands): primary mode is Ocean container / bulk. Largest port in Europe and key gateway to the EU hinterland. Shipments routed through Port of Rotterdam should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Port of Antwerp-Bruges',
     'content': 'Port profile — Port of Antwerp-Bruges (Belgium): primary mode is Ocean container / bulk / chemicals. Second-largest European port, major petrochemical cluster. Shipments routed through Port of Antwerp-Bruges should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Port of Hamburg',
     'content': "Port profile — Port of Hamburg (Germany): primary mode is Ocean container. Germany's largest port, key rail-connected gateway to Central Europe. Shipments routed through Port of Hamburg should account for local congestion trends and seasonal weather risk."},
    {'title': 'Port Profile: Port of Los Angeles',
     'content': 'Port profile — Port of Los Angeles (United States): primary mode is Ocean container. Busiest container port in North America by volume. Shipments routed through Port of Los Angeles should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Port of Long Beach',
     'content': 'Port profile — Port of Long Beach (United States): primary mode is Ocean container. Adjacent to Los Angeles, forms the largest US container complex. Shipments routed through Port of Long Beach should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Port of New York and New Jersey',
     'content': 'Port profile — Port of New York and New Jersey (United States): primary mode is Ocean container. Largest port on the US East Coast. Shipments routed through Port of New York and New Jersey should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Port of Savannah',
     'content': 'Port profile — Port of Savannah (United States): primary mode is Ocean container. Fastest-growing major US container port in recent years. Shipments routed through Port of Savannah should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Port of Nhava Sheva / JNPT',
     'content': "Port profile — Port of Nhava Sheva / JNPT (India): primary mode is Ocean container. India's largest container port by volume. Shipments routed through Port of Nhava Sheva / JNPT should account for local congestion trends and seasonal weather risk."},
    {'title': 'Port Profile: Mundra Port',
     'content': "Port profile — Mundra Port (India): primary mode is Ocean container / bulk. India's largest private port operator, on the west coast. Shipments routed through Mundra Port should account for local congestion trends and seasonal weather risk."},
    {'title': 'Port Profile: Chennai Port',
     'content': 'Port profile — Chennai Port (India): primary mode is Ocean container / bulk. Major gateway for South India and automotive exports. Shipments routed through Chennai Port should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Port Klang',
     'content': "Port profile — Port Klang (Malaysia): primary mode is Ocean container / transshipment. Malaysia's principal gateway port. Shipments routed through Port Klang should account for local congestion trends and seasonal weather risk."},
    {'title': 'Port Profile: Tanjung Pelepas Port',
     'content': 'Port profile — Tanjung Pelepas Port (Malaysia): primary mode is Ocean container / transshipment. Deep-water transshipment hub near the Strait of Malacca. Shipments routed through Tanjung Pelepas Port should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Laem Chabang Port',
     'content': "Port profile — Laem Chabang Port (Thailand): primary mode is Ocean container. Thailand's main deep-sea container port. Shipments routed through Laem Chabang Port should account for local congestion trends and seasonal weather risk."},
    {'title': 'Port Profile: Tanjung Priok Port',
     'content': "Port profile — Tanjung Priok Port (Indonesia): primary mode is Ocean container. Indonesia's busiest port, serving greater Jakarta. Shipments routed through Tanjung Priok Port should account for local congestion trends and seasonal weather risk."},
    {'title': 'Port Profile: Port of Colombo',
     'content': 'Port profile — Port of Colombo (Sri Lanka): primary mode is Ocean container / transshipment. Key transshipment hub for South Asia. Shipments routed through Port of Colombo should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Piraeus Port',
     'content': 'Port profile — Piraeus Port (Greece): primary mode is Ocean container / transshipment. Mediterranean transshipment gateway to Southeast Europe. Shipments routed through Piraeus Port should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Port of Valencia',
     'content': "Port profile — Port of Valencia (Spain): primary mode is Ocean container. Largest container port on the Mediterranean's western coast. Shipments routed through Port of Valencia should account for local congestion trends and seasonal weather risk."},
    {'title': 'Port Profile: Port of Felixstowe',
     'content': "Port profile — Port of Felixstowe (United Kingdom): primary mode is Ocean container. UK's busiest container port. Shipments routed through Port of Felixstowe should account for local congestion trends and seasonal weather risk."},
    {'title': 'Port Profile: Port of Santos',
     'content': 'Port profile — Port of Santos (Brazil): primary mode is Ocean container / bulk. Largest port in Latin America. Shipments routed through Port of Santos should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Port of Durban',
     'content': 'Port profile — Port of Durban (South Africa): primary mode is Ocean container / bulk. Busiest container port in Sub-Saharan Africa. Shipments routed through Port of Durban should account for local congestion trends and seasonal weather risk.'},
    {'title': 'Port Profile: Port of Vancouver',
     'content': "Port profile — Port of Vancouver (Canada): primary mode is Ocean container / bulk. Canada's largest and most diversified port. Shipments routed through Port of Vancouver should account for local congestion trends and seasonal weather risk."},
    {'title': 'HS Chapter 01 Guide',
     'content': 'HS Chapter 01: Live animals. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 02 Guide',
     'content': 'HS Chapter 02: Meat and edible meat offal. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 03 Guide',
     'content': 'HS Chapter 03: Fish and crustaceans, molluscs and other aquatic invertebrates. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 04 Guide',
     'content': "HS Chapter 04: Dairy produce; birds' eggs; natural honey. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide."},
    {'title': 'HS Chapter 07 Guide',
     'content': 'HS Chapter 07: Edible vegetables and certain roots and tubers. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 08 Guide',
     'content': 'HS Chapter 08: Edible fruit and nuts; peel of citrus fruit or melons. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 09 Guide',
     'content': 'HS Chapter 09: Coffee, tea, mate and spices. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 10 Guide',
     'content': 'HS Chapter 10: Cereals. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 15 Guide',
     'content': 'HS Chapter 15: Animal, vegetable or microbial fats and oils. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 17 Guide',
     'content': 'HS Chapter 17: Sugars and sugar confectionery. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 22 Guide',
     'content': 'HS Chapter 22: Beverages, spirits and vinegar. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 25 Guide',
     'content': 'HS Chapter 25: Salt; sulphur; earths and stone; plastering materials, lime and cement. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 27 Guide',
     'content': 'HS Chapter 27: Mineral fuels, mineral oils and products of their distillation; bituminous substances; mineral waxes. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 28 Guide',
     'content': 'HS Chapter 28: Inorganic chemicals; organic or inorganic compounds of precious metals. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 29 Guide',
     'content': 'HS Chapter 29: Organic chemicals. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 33 Guide',
     'content': 'HS Chapter 33: Essential oils and resinoids; perfumery, cosmetic or toilet preparations. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 39 Guide',
     'content': 'HS Chapter 39: Plastics and articles thereof. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 40 Guide',
     'content': 'HS Chapter 40: Rubber and articles thereof. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 44 Guide',
     'content': 'HS Chapter 44: Wood and articles of wood; wood charcoal. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 48 Guide',
     'content': 'HS Chapter 48: Paper and paperboard; articles of paper pulp, of paper or of paperboard. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 64 Guide',
     'content': 'HS Chapter 64: Footwear, gaiters and the like; parts of such articles. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 68 Guide',
     'content': 'HS Chapter 68: Articles of stone, plaster, cement, asbestos, mica or similar materials. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 69 Guide',
     'content': 'HS Chapter 69: Ceramic products. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 70 Guide',
     'content': 'HS Chapter 70: Glass and glassware. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 71 Guide',
     'content': 'HS Chapter 71: Natural or cultured pearls, precious or semi-precious stones, precious metals. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 72 Guide',
     'content': 'HS Chapter 72: Iron and steel. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 73 Guide',
     'content': 'HS Chapter 73: Articles of iron or steel. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 76 Guide',
     'content': 'HS Chapter 76: Aluminium and articles thereof. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 84 Guide',
     'content': 'HS Chapter 84: Nuclear reactors, boilers, machinery and mechanical appliances; parts thereof. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 86 Guide',
     'content': 'HS Chapter 86: Railway or tramway locomotives, rolling-stock and parts thereof. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 87 Guide',
     'content': 'HS Chapter 87: Vehicles other than railway or tramway rolling-stock, and parts and accessories thereof. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 88 Guide',
     'content': 'HS Chapter 88: Aircraft, spacecraft, and parts thereof. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 89 Guide',
     'content': 'HS Chapter 89: Ships, boats and floating structures. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 90 Guide',
     'content': 'HS Chapter 90: Optical, photographic, measuring, checking, precision, medical or surgical instruments. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'HS Chapter 94 Guide',
     'content': 'HS Chapter 94: Furniture; bedding, mattresses; lamps and lighting fittings; illuminated signs; prefabricated buildings. This chapter of the Harmonized System nomenclature is used to classify products in customs declarations, commercial invoices, and tariff schedules worldwide.'},
    {'title': 'Trade Corridor Brief: China to United States',
     'content': 'This brief summarizes the China-to-United States freight corridor. The dominant transport mode on this lane is Ocean FCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the China-United States lane, monsoon-season congestion at transshipment ports typically adds 2-4 days of transit variability. Base freight cost estimates for Ocean FCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: China to Germany',
     'content': 'This brief summarizes the China-to-Germany freight corridor. The dominant transport mode on this lane is Ocean LCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the China-Germany lane, peak-season surcharges (PSS) commonly apply from August through October. Base freight cost estimates for Ocean LCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: China to India',
     'content': 'This brief summarizes the China-to-India freight corridor. The dominant transport mode on this lane is Air Freight. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the China-India lane, customs pre-clearance programs can reduce dwell time by up to 30% for AEO-certified shippers. Base freight cost estimates for Air Freight on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: India to United Arab Emirates',
     'content': 'This brief summarizes the India-to-United Arab Emirates freight corridor. The dominant transport mode on this lane is Rail/Intermodal. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the India-United Arab Emirates lane, container availability tightens ahead of major regional holidays, so early booking is advised. Base freight cost estimates for Rail/Intermodal on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: India to United States',
     'content': 'This brief summarizes the India-to-United States freight corridor. The dominant transport mode on this lane is Ocean FCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the India-United States lane, geopolitical or canal/strait disruptions on this corridor should be monitored via the Route Optimization Agent. Base freight cost estimates for Ocean FCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: India to United Kingdom',
     'content': 'This brief summarizes the India-to-United Kingdom freight corridor. The dominant transport mode on this lane is Ocean LCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the India-United Kingdom lane, monsoon-season congestion at transshipment ports typically adds 2-4 days of transit variability. Base freight cost estimates for Ocean LCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: United States to Mexico',
     'content': 'This brief summarizes the United States-to-Mexico freight corridor. The dominant transport mode on this lane is Air Freight. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the United States-Mexico lane, peak-season surcharges (PSS) commonly apply from August through October. Base freight cost estimates for Air Freight on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: United States to Canada',
     'content': 'This brief summarizes the United States-to-Canada freight corridor. The dominant transport mode on this lane is Rail/Intermodal. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the United States-Canada lane, customs pre-clearance programs can reduce dwell time by up to 30% for AEO-certified shippers. Base freight cost estimates for Rail/Intermodal on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: United States to Brazil',
     'content': 'This brief summarizes the United States-to-Brazil freight corridor. The dominant transport mode on this lane is Ocean FCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the United States-Brazil lane, container availability tightens ahead of major regional holidays, so early booking is advised. Base freight cost estimates for Ocean FCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Germany to Poland',
     'content': 'This brief summarizes the Germany-to-Poland freight corridor. The dominant transport mode on this lane is Ocean LCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Germany-Poland lane, geopolitical or canal/strait disruptions on this corridor should be monitored via the Route Optimization Agent. Base freight cost estimates for Ocean LCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Germany to France',
     'content': 'This brief summarizes the Germany-to-France freight corridor. The dominant transport mode on this lane is Air Freight. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Germany-France lane, monsoon-season congestion at transshipment ports typically adds 2-4 days of transit variability. Base freight cost estimates for Air Freight on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Netherlands to United Kingdom',
     'content': 'This brief summarizes the Netherlands-to-United Kingdom freight corridor. The dominant transport mode on this lane is Rail/Intermodal. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Netherlands-United Kingdom lane, peak-season surcharges (PSS) commonly apply from August through October. Base freight cost estimates for Rail/Intermodal on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Singapore to Malaysia',
     'content': 'This brief summarizes the Singapore-to-Malaysia freight corridor. The dominant transport mode on this lane is Ocean FCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Singapore-Malaysia lane, customs pre-clearance programs can reduce dwell time by up to 30% for AEO-certified shippers. Base freight cost estimates for Ocean FCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Singapore to Indonesia',
     'content': 'This brief summarizes the Singapore-to-Indonesia freight corridor. The dominant transport mode on this lane is Ocean LCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Singapore-Indonesia lane, container availability tightens ahead of major regional holidays, so early booking is advised. Base freight cost estimates for Ocean LCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Singapore to Australia',
     'content': 'This brief summarizes the Singapore-to-Australia freight corridor. The dominant transport mode on this lane is Air Freight. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Singapore-Australia lane, geopolitical or canal/strait disruptions on this corridor should be monitored via the Route Optimization Agent. Base freight cost estimates for Air Freight on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Japan to South Korea',
     'content': 'This brief summarizes the Japan-to-South Korea freight corridor. The dominant transport mode on this lane is Rail/Intermodal. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Japan-South Korea lane, monsoon-season congestion at transshipment ports typically adds 2-4 days of transit variability. Base freight cost estimates for Rail/Intermodal on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Japan to Vietnam',
     'content': 'This brief summarizes the Japan-to-Vietnam freight corridor. The dominant transport mode on this lane is Ocean FCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Japan-Vietnam lane, peak-season surcharges (PSS) commonly apply from August through October. Base freight cost estimates for Ocean FCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: South Korea to China',
     'content': 'This brief summarizes the South Korea-to-China freight corridor. The dominant transport mode on this lane is Ocean LCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the South Korea-China lane, customs pre-clearance programs can reduce dwell time by up to 30% for AEO-certified shippers. Base freight cost estimates for Ocean LCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Vietnam to United States',
     'content': 'This brief summarizes the Vietnam-to-United States freight corridor. The dominant transport mode on this lane is Air Freight. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Vietnam-United States lane, container availability tightens ahead of major regional holidays, so early booking is advised. Base freight cost estimates for Air Freight on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Vietnam to Japan',
     'content': 'This brief summarizes the Vietnam-to-Japan freight corridor. The dominant transport mode on this lane is Rail/Intermodal. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Vietnam-Japan lane, geopolitical or canal/strait disruptions on this corridor should be monitored via the Route Optimization Agent. Base freight cost estimates for Rail/Intermodal on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Thailand to China',
     'content': 'This brief summarizes the Thailand-to-China freight corridor. The dominant transport mode on this lane is Ocean FCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Thailand-China lane, monsoon-season congestion at transshipment ports typically adds 2-4 days of transit variability. Base freight cost estimates for Ocean FCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Indonesia to China',
     'content': 'This brief summarizes the Indonesia-to-China freight corridor. The dominant transport mode on this lane is Ocean LCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Indonesia-China lane, peak-season surcharges (PSS) commonly apply from August through October. Base freight cost estimates for Ocean LCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Malaysia to China',
     'content': 'This brief summarizes the Malaysia-to-China freight corridor. The dominant transport mode on this lane is Air Freight. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Malaysia-China lane, customs pre-clearance programs can reduce dwell time by up to 30% for AEO-certified shippers. Base freight cost estimates for Air Freight on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Philippines to Japan',
     'content': 'This brief summarizes the Philippines-to-Japan freight corridor. The dominant transport mode on this lane is Rail/Intermodal. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Philippines-Japan lane, container availability tightens ahead of major regional holidays, so early booking is advised. Base freight cost estimates for Rail/Intermodal on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: United Arab Emirates to India',
     'content': 'This brief summarizes the United Arab Emirates-to-India freight corridor. The dominant transport mode on this lane is Ocean FCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the United Arab Emirates-India lane, geopolitical or canal/strait disruptions on this corridor should be monitored via the Route Optimization Agent. Base freight cost estimates for Ocean FCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: United Arab Emirates to Saudi Arabia',
     'content': 'This brief summarizes the United Arab Emirates-to-Saudi Arabia freight corridor. The dominant transport mode on this lane is Ocean LCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the United Arab Emirates-Saudi Arabia lane, monsoon-season congestion at transshipment ports typically adds 2-4 days of transit variability. Base freight cost estimates for Ocean LCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Saudi Arabia to China',
     'content': 'This brief summarizes the Saudi Arabia-to-China freight corridor. The dominant transport mode on this lane is Air Freight. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Saudi Arabia-China lane, peak-season surcharges (PSS) commonly apply from August through October. Base freight cost estimates for Air Freight on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Turkey to Germany',
     'content': 'This brief summarizes the Turkey-to-Germany freight corridor. The dominant transport mode on this lane is Rail/Intermodal. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Turkey-Germany lane, customs pre-clearance programs can reduce dwell time by up to 30% for AEO-certified shippers. Base freight cost estimates for Rail/Intermodal on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Turkey to United Kingdom',
     'content': 'This brief summarizes the Turkey-to-United Kingdom freight corridor. The dominant transport mode on this lane is Ocean FCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Turkey-United Kingdom lane, container availability tightens ahead of major regional holidays, so early booking is advised. Base freight cost estimates for Ocean FCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Egypt to European Union',
     'content': 'This brief summarizes the Egypt-to-European Union freight corridor. The dominant transport mode on this lane is Ocean LCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Egypt-European Union lane, geopolitical or canal/strait disruptions on this corridor should be monitored via the Route Optimization Agent. Base freight cost estimates for Ocean LCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: South Africa to China',
     'content': 'This brief summarizes the South Africa-to-China freight corridor. The dominant transport mode on this lane is Air Freight. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the South Africa-China lane, monsoon-season congestion at transshipment ports typically adds 2-4 days of transit variability. Base freight cost estimates for Air Freight on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Nigeria to China',
     'content': 'This brief summarizes the Nigeria-to-China freight corridor. The dominant transport mode on this lane is Rail/Intermodal. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Nigeria-China lane, peak-season surcharges (PSS) commonly apply from August through October. Base freight cost estimates for Rail/Intermodal on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Kenya to United Arab Emirates',
     'content': 'This brief summarizes the Kenya-to-United Arab Emirates freight corridor. The dominant transport mode on this lane is Ocean FCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Kenya-United Arab Emirates lane, customs pre-clearance programs can reduce dwell time by up to 30% for AEO-certified shippers. Base freight cost estimates for Ocean FCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Brazil to China',
     'content': 'This brief summarizes the Brazil-to-China freight corridor. The dominant transport mode on this lane is Ocean LCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Brazil-China lane, container availability tightens ahead of major regional holidays, so early booking is advised. Base freight cost estimates for Ocean LCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Brazil to United States',
     'content': 'This brief summarizes the Brazil-to-United States freight corridor. The dominant transport mode on this lane is Air Freight. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Brazil-United States lane, geopolitical or canal/strait disruptions on this corridor should be monitored via the Route Optimization Agent. Base freight cost estimates for Air Freight on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Argentina to Brazil',
     'content': 'This brief summarizes the Argentina-to-Brazil freight corridor. The dominant transport mode on this lane is Rail/Intermodal. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Argentina-Brazil lane, monsoon-season congestion at transshipment ports typically adds 2-4 days of transit variability. Base freight cost estimates for Rail/Intermodal on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Chile to China',
     'content': 'This brief summarizes the Chile-to-China freight corridor. The dominant transport mode on this lane is Ocean FCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Chile-China lane, peak-season surcharges (PSS) commonly apply from August through October. Base freight cost estimates for Ocean FCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Colombia to United States',
     'content': 'This brief summarizes the Colombia-to-United States freight corridor. The dominant transport mode on this lane is Ocean LCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Colombia-United States lane, customs pre-clearance programs can reduce dwell time by up to 30% for AEO-certified shippers. Base freight cost estimates for Ocean LCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Mexico to China',
     'content': 'This brief summarizes the Mexico-to-China freight corridor. The dominant transport mode on this lane is Air Freight. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Mexico-China lane, container availability tightens ahead of major regional holidays, so early booking is advised. Base freight cost estimates for Air Freight on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Canada to China',
     'content': 'This brief summarizes the Canada-to-China freight corridor. The dominant transport mode on this lane is Rail/Intermodal. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Canada-China lane, geopolitical or canal/strait disruptions on this corridor should be monitored via the Route Optimization Agent. Base freight cost estimates for Rail/Intermodal on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Australia to China',
     'content': 'This brief summarizes the Australia-to-China freight corridor. The dominant transport mode on this lane is Ocean FCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Australia-China lane, monsoon-season congestion at transshipment ports typically adds 2-4 days of transit variability. Base freight cost estimates for Ocean FCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Australia to Japan',
     'content': 'This brief summarizes the Australia-to-Japan freight corridor. The dominant transport mode on this lane is Ocean LCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Australia-Japan lane, peak-season surcharges (PSS) commonly apply from August through October. Base freight cost estimates for Ocean LCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: New Zealand to China',
     'content': 'This brief summarizes the New Zealand-to-China freight corridor. The dominant transport mode on this lane is Air Freight. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the New Zealand-China lane, customs pre-clearance programs can reduce dwell time by up to 30% for AEO-certified shippers. Base freight cost estimates for Air Freight on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Israel to United States',
     'content': 'This brief summarizes the Israel-to-United States freight corridor. The dominant transport mode on this lane is Rail/Intermodal. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Israel-United States lane, container availability tightens ahead of major regional holidays, so early booking is advised. Base freight cost estimates for Rail/Intermodal on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Bangladesh to European Union',
     'content': 'This brief summarizes the Bangladesh-to-European Union freight corridor. The dominant transport mode on this lane is Ocean FCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Bangladesh-European Union lane, geopolitical or canal/strait disruptions on this corridor should be monitored via the Route Optimization Agent. Base freight cost estimates for Ocean FCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Pakistan to China',
     'content': 'This brief summarizes the Pakistan-to-China freight corridor. The dominant transport mode on this lane is Ocean LCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Pakistan-China lane, monsoon-season congestion at transshipment ports typically adds 2-4 days of transit variability. Base freight cost estimates for Ocean LCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Sweden to Germany',
     'content': 'This brief summarizes the Sweden-to-Germany freight corridor. The dominant transport mode on this lane is Air Freight. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Sweden-Germany lane, peak-season surcharges (PSS) commonly apply from August through October. Base freight cost estimates for Air Freight on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Switzerland to European Union',
     'content': 'This brief summarizes the Switzerland-to-European Union freight corridor. The dominant transport mode on this lane is Rail/Intermodal. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Switzerland-European Union lane, customs pre-clearance programs can reduce dwell time by up to 30% for AEO-certified shippers. Base freight cost estimates for Rail/Intermodal on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Ireland to United States',
     'content': 'This brief summarizes the Ireland-to-United States freight corridor. The dominant transport mode on this lane is Ocean FCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Ireland-United States lane, container availability tightens ahead of major regional holidays, so early booking is advised. Base freight cost estimates for Ocean FCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Qatar to European Union',
     'content': 'This brief summarizes the Qatar-to-European Union freight corridor. The dominant transport mode on this lane is Ocean LCL. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Qatar-European Union lane, geopolitical or canal/strait disruptions on this corridor should be monitored via the Route Optimization Agent. Base freight cost estimates for Ocean LCL on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
    {'title': 'Trade Corridor Brief: Poland to Ukraine',
     'content': 'This brief summarizes the Poland-to-Ukraine freight corridor. The dominant transport mode on this lane is Air Freight. Freight rates on this corridor are influenced by fuel surcharges (BAF), port congestion, and seasonal demand swings. Shipments typically require a Commercial Invoice, Packing List, Bill of Lading or Air Waybill, Certificate of Origin (where a trade agreement applies), and an HS-code-accurate customs declaration. On the Poland-Ukraine lane, monsoon-season congestion at transshipment ports typically adds 2-4 days of transit variability. Base freight cost estimates for Air Freight on this corridor should be adjusted for the current Bunker Adjustment Factor (BAF) and, where applicable, a Peak Season Surcharge (PSS).'},
])

print(f'Merged in {len(FREIGHT_DOCS)} total freight knowledge documents (base 25 + merged extras from the RAG Pipeline notebook).')

doc_path = f'{KB_DIR}/freight_kb.jsonl'
with open(doc_path, 'w') as f:
    for doc in FREIGHT_DOCS:
        f.write(json.dumps(doc) + '\n')

print(f'Generated {len(FREIGHT_DOCS)} freight knowledge documents -> Drive')
categories = {'INCOTERMS': 6, 'HS Codes': 4, 'Customs Regs': 5, 'Port Ops': 4, 'Pricing': 5, 'Carriers': 3, 'Strategy': 2}
for cat, n in categories.items():
    print(f'  {cat}: {n} docs')


In [ ]:
import json, numpy as np, faiss, pickle, os, time
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

def chunk_text(text, size=350, overlap=70):
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunks.append(' '.join(words[i:i+size]))
        i += size - overlap
    return chunks

# Load all KB files
all_chunks, cid = [], 0
for fname in ['freight_kb.jsonl', 'pdf_extracted.jsonl']:
    fpath = f'{KB_DIR}/{fname}'
    if not os.path.exists(fpath): continue
    with open(fpath) as f:
        for line in f:
            doc = json.loads(line)
            for chunk in chunk_text(doc['content']):
                all_chunks.append({'chunk_id': f'fq_{cid:05d}', 'title': doc['title'],
                                    'text': chunk, 'source': fname.replace('.jsonl',''),
                                    'words': len(chunk.split())})
                cid += 1

chunks_path = f'{KB_DIR}/all_chunks.jsonl'
with open(chunks_path, 'w') as f:
    [f.write(json.dumps(c) + '\n') for c in all_chunks]
print(f'Chunks: {len(all_chunks)} -> {chunks_path}')

# ── FAISS Dense Index ─────────────────────────────────────────────────────
t0 = time.time()
embedder = SentenceTransformer('all-MiniLM-L6-v2', cache_folder=ST_CACHE)
texts = [c['text'] for c in all_chunks]
embeddings = embedder.encode(texts, batch_size=128, show_progress_bar=True, convert_to_numpy=True).astype('float32')
faiss.normalize_L2(embeddings)
dim = embeddings.shape[1]
nlist = min(32, len(all_chunks) // 4)
quantizer = faiss.IndexFlatIP(dim)
index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
index.train(embeddings)
index.add(embeddings)
index.nprobe = 8
faiss.write_index(index, f'{FAISS_DIR}/freight_faiss.index')
with open(f'{FAISS_DIR}/freight_chunks_meta.json', 'w') as f:
    json.dump(all_chunks, f)
print(f'FAISS: {index.ntotal} vectors, dim={dim} -> Drive ({time.time()-t0:.1f}s)')

# ── BM25 Sparse Index ─────────────────────────────────────────────────────
tokenized = [c['text'].lower().split() for c in all_chunks]
bm25 = BM25Okapi(tokenized)
with open(f'{BM25_DIR}/freight_bm25.pkl', 'wb') as f:
    pickle.dump({'bm25': bm25, 'chunks': all_chunks}, f)
print(f'BM25: {len(all_chunks)} docs -> Drive')


Chunks: 29 -> /content/drive/MyDrive/FreightQuote_AI/synthetic_kb/all_chunks.jsonl


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS: 29 vectors, dim=384 -> Drive (13.0s)
BM25: 29 docs -> Drive


In [ ]:
import json, numpy as np, faiss, pickle, time
from sentence_transformers import SentenceTransformer

class FreightRAG:
    def __init__(self):
        t0 = time.time()
        self.index    = faiss.read_index(f'{FAISS_DIR}/freight_faiss.index')
        self.index.nprobe = 8
        with open(f'{FAISS_DIR}/freight_chunks_meta.json') as f:
            self.chunks = json.load(f)
        with open(f'{BM25_DIR}/freight_bm25.pkl', 'rb') as f:
            data = pickle.load(f)
        self.bm25     = data['bm25']
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2', cache_folder=ST_CACHE)
        print(f'FreightRAG ready in {time.time()-t0:.1f}s | {len(self.chunks)} chunks')

    def search(self, query, k=4, alpha=0.6):
        q_emb = self.embedder.encode([query], convert_to_numpy=True).astype('float32')
        faiss.normalize_L2(q_emb)
        scores, idxs = self.index.search(q_emb, k*2)
        dense = [{'chunk': self.chunks[i], 'ds': float(s)} for s, i in zip(scores[0], idxs[0]) if i >= 0]
        tokens = query.lower().split()
        bm25_s = self.bm25.get_scores(tokens)
        top_k  = np.argsort(bm25_s)[::-1][:k*2]
        sparse = [{'chunk': self.chunks[i], 'bs': float(bm25_s[i])} for i in top_k if bm25_s[i] > 0]
        scored = {}
        if dense:
            dm = max(c['ds'] for c in dense) or 1.0
            for c in dense:
                cid = c['chunk']['chunk_id']
                scored[cid] = scored.get(cid, {'chunk': c['chunk'], 'score': 0.0})
                scored[cid]['score'] += alpha * (c['ds'] / dm)
        if sparse:
            bm = max(c['bs'] for c in sparse) or 1.0
            for c in sparse:
                cid = c['chunk']['chunk_id']
                scored[cid] = scored.get(cid, {'chunk': c['chunk'], 'score': 0.0})
                scored[cid]['score'] += (1-alpha) * (c['bs'] / bm)
        ranked = sorted(scored.values(), key=lambda x: x['score'], reverse=True)[:k]
        return [{'text': r['chunk']['text'], 'title': r['chunk']['title'],
                 'source': r['chunk']['source'], 'score': r['score']} for r in ranked]

rag_fq = FreightRAG()

FQ_QUERIES = [
    'What is the difference between FOB and CIF?',
    'How are dangerous goods classified for shipping?',
    'What documents required for US customs clearance?',
    'How is freight CO2 emissions calculated?',
    'What are peak season surcharges?',
    'Which ports have lowest congestion?',
    'How do I calculate cargo insurance premium?',
]

print('FreightQuote RAG Test Queries:')
for q in FQ_QUERIES:
    results = rag_fq.search(q, k=1)
    r = results[0]
    print(f'  Q: {q[:45]:47} Score:{r["score"]:.3f} [{r["title"][:25]}]')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FreightRAG ready in 0.8s | 29 chunks
FreightQuote RAG Test Queries:
  Q: What is the difference between FOB and CIF?     Score:1.000 [INCOTERMS-FOB: Free on Bo]
  Q: How are dangerous goods classified for shippi   Score:1.000 [HS Code Classification: D]
  Q: What documents required for US customs cleara   Score:1.000 [Customs Regulation: Unite]
  Q: How is freight CO2 emissions calculated?        Score:1.000 [Freight Pricing: CO2 Emis]
  Q: What are peak season surcharges?                Score:1.000 [Freight Pricing: Surcharg]
  Q: Which ports have lowest congestion?             Score:0.923 [Port Operations: Singapor]
  Q: How do I calculate cargo insurance premium?     Score:1.000 [Freight Pricing: Insuranc]


In [ ]:
FQ_RAG_MODULE = '''
import os, json, pickle, numpy as np

_retriever = None
FAISS_DIR = '/content/drive/MyDrive/FreightQuote_AI/faiss_indexes'
BM25_DIR  = '/content/drive/MyDrive/FreightQuote_AI/bm25_indexes'
ST_CACHE  = '/content/.cache/sentence_transformers'

def _load_retriever():
    global _retriever
    if _retriever is not None: return _retriever
    try:
        import faiss
        from sentence_transformers import SentenceTransformer
        idx = faiss.read_index(f'{FAISS_DIR}/freight_faiss.index')
        idx.nprobe = 8
        with open(f'{FAISS_DIR}/freight_chunks_meta.json') as f: chunks = json.load(f)
        with open(f'{BM25_DIR}/freight_bm25.pkl', 'rb') as f: data = pickle.load(f)
        emb = SentenceTransformer('all-MiniLM-L6-v2', cache_folder=ST_CACHE)
        _retriever = {'index': idx, 'chunks': chunks, 'bm25': data['bm25'], 'embedder': emb}
        return _retriever
    except:
        return None

def is_rag_ready():
    return os.path.exists(f'{FAISS_DIR}/freight_faiss.index')

def retrieve(query, k=4, alpha=0.6):
    r = _load_retriever()
    if r is None:
        return [{'text': 'Run FreightQuote_RAG_Builder.ipynb first.', 'title': 'System', 'source': 'system', 'score': 0}]
    import faiss
    q_emb = r['embedder'].encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(q_emb)
    scores, idxs = r['index'].search(q_emb, k*2)
    dense = [{'chunk': r['chunks'][i], 'ds': float(s)} for s, i in zip(scores[0], idxs[0]) if i >= 0]
    tokens = query.lower().split()
    bm25_s = r['bm25'].get_scores(tokens)
    top_k  = np.argsort(bm25_s)[::-1][:k*2]
    sparse = [{'chunk': r['chunks'][i], 'bs': float(bm25_s[i])} for i in top_k if bm25_s[i] > 0]
    scored = {}
    if dense:
        dm = max(c['ds'] for c in dense) or 1.0
        for c in dense:
            cid = c['chunk']['chunk_id']
            scored[cid] = scored.get(cid, {'chunk': c['chunk'], 'score': 0.0})
            scored[cid]['score'] += alpha * (c['ds'] / dm)
    if sparse:
        bm = max(c['bs'] for c in sparse) or 1.0
        for c in sparse:
            cid = c['chunk']['chunk_id']
            scored[cid] = scored.get(cid, {'chunk': c['chunk'], 'score': 0.0})
            scored[cid]['score'] += (1-alpha) * (c['bs'] / bm)
    ranked = sorted(scored.values(), key=lambda x: x['score'], reverse=True)[:k]
    return [{'text': r['chunk']['text'], 'title': r['chunk']['title'], 'source': r['chunk']['source'], 'score': r['score']} for r in ranked]

def answer_with_citation(query):
    results = retrieve(query)
    if not results or results[0]['score'] == 0:
        return 'No context found.', 'None'
    return ' '.join([r['text'] for r in results]), results[0]['title']
'''

with open('/content/rag_engine_freight.py', 'w') as f:
    f.write(FQ_RAG_MODULE)
print('Exported: /content/rag_engine_freight.py')


Exported: /content/rag_engine_freight.py


In [ ]:
import plotly.express as px, pandas as pd, os

FQ_QUERIES = [
    'What is the difference between FOB and CIF?',
    'How are dangerous goods classified for shipping?',
    'What documents required for US customs clearance?',
    'How is freight CO2 calculated?',
    'What are peak season surcharges?',
    'How to calculate cargo insurance premium?',
    'Which INCOTERM gives seller minimum obligation?',
    'What is the HS code for smartphones?',
]
# ── Additional test queries merged in from FreightQuote_AI_RAG_Pipeline.ipynb (new items only, de-duplicated against FQ_QUERIES above) ──
FQ_QUERIES.extend([
    'What are the customs clearance steps for importing electronics into the US?',
    'What is ISF 10+2 and when must it be filed?',
    'What Incoterm makes the seller responsible for all delivery costs?',
    'What is the sulphur limit under IMO 2020?',
    'What documents are mandatory on a Bill of Lading?',
    'What is the operational cost formula for freight margin calculation?',
    'When does a freight quote require human approval?',
    'What does FOB mean in shipping terms?',
    'What does EXW mean?',
    'What is a Certificate of Origin used for?',
    'What are demurrage charges?',
    'What is the difference between demurrage and detention?',
    'What is a TEU?',
    'What is a Bunker Adjustment Factor?',
    'What is Harmonized System HS code chapter 85 about?',
    'What is HS Chapter 87 used for?',
    'What products fall under HS Chapter 30?',
    'What is the standard VAT rate in Germany?',
    'What is the customs authority in India?',
    'What GST rate applies in Singapore?',
    'Which authority handles customs in the United Arab Emirates?',
    'What is the busiest container port in the world?',
    'What role does Jebel Ali Port play in trade?',
    'Which port is the largest in Latin America?',
    'What risks affect the China to United States trade corridor?',
    'What documents are required on the India to United Arab Emirates corridor?',
    'What is a Letter of Credit used for?',
    'What is an ATA Carnet?',
    'What are anti-dumping duties?',
    'What does chargeable weight mean in air freight?',
    'What is the difference between FCL and LCL?',
    'What is a bonded warehouse?',
    'What does AEO status provide to traders?',
    'What triggers a carrier audit flag in FreightQuote AI?',
    'How are carrier tiers classified in FreightQuote AI?',
    'What wind speed triggers weather-based re-routing?',
    'What is the Carbon Border Adjustment Mechanism (CBAM)?',
    'What is SOLAS VGM?',
    'What is a Non-Vessel-Operating Common Carrier (NVOCC)?',
    'What is General Average in maritime law?',
])

all_results = []
for q in FQ_QUERIES:
    for r in rag_fq.search(q, k=3):
        all_results.append({'Query': q[:35]+'..', 'Title': r['title'][:28], 'Score': r['score'], 'Source': r['source']})
df = pd.DataFrame(all_results)
fig = px.bar(df.groupby('Source')['Score'].mean().reset_index(),
    x='Source', y='Score', color='Source', title='FreightQuote RAG: Avg Score by Source')
fig.show()

print('=' * 60)
print('  FREIGHTQUOTE RAG BUILDER - COMPLETE')
print('=' * 60)
files = [
    (f'{FAISS_DIR}/freight_faiss.index', 'FAISS dense index'),
    (f'{FAISS_DIR}/freight_chunks_meta.json', 'Chunk metadata'),
    (f'{BM25_DIR}/freight_bm25.pkl', 'BM25 sparse index'),
    (f'{KB_DIR}/freight_kb.jsonl', 'Freight knowledge base'),
    ('/content/rag_engine_freight.py', 'RAG engine module'),
]
for path, desc in files:
    exists = os.path.exists(path)
    size = os.path.getsize(path)/1024 if exists else 0
    print(f'  {chr(9989) if exists else chr(10060)} {desc}: {path.split("/")[-1]} ({size:.1f} KB)')
print()
print('Drive-backed files persist between Colab sessions!')
print('LLM stays local to save Drive quota.')
print()
print('DRIVE STORAGE ESTIMATE:')
print('  FAISS index:     ~2-10 MB depending on chunk count')
print('  BM25 index:      ~1-5 MB')
print('  ML joblib models: ~5-20 MB each')
print('  LLM (local only): ~2-5 GB - NEVER stored in Drive')
print(f'Avg retrieval score: {df["Score"].mean():.3f}')
